[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab07_document_processing.ipynb)

# Lab 7: การประมวลผลเอกสารและการสกัดสารสนเทศทางคลินิก (Document Processing & Clinical Information Extraction)

**รายวิชา:** 3099704 AI for Digital Health (2026) · **คาบเรียน:** วันเสาร์ที่ 17 ตุลาคม 2026 (ช่วงเช้า) · **เวลาโดยประมาณ:** 60–90 นาที

## วัตถุประสงค์การเรียนรู้

- สกัดข้อความจากเอกสารทางคลินิกที่เป็นภาพสแกนหรือ PDF ด้วยการรู้จำอักขระด้วยแสง (OCR) โดยใช้ Tesseract
- ทำความสะอาดและจัดโครงสร้างข้อความจากเอกสารที่ไม่เป็นระเบียบ
- สกัดสารสนเทศทางคลินิก (ผลการตรวจทางห้องปฏิบัติการ และรายการยา) ด้วยกฎ (rules) และอธิบายประโยชน์ของโมเดลประมวลผลภาษาธรรมชาติ (NLP) ที่เรียนรู้จากข้อมูล
- สร้างระเบียนข้อมูลแบบมีโครงสร้าง (structured record) จากเอกสารที่ไม่มีโครงสร้าง และประเมินความถูกต้องของผลลัพธ์

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
>
> - ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม:** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง โดยคลิกที่เซลล์แล้วกด **Shift+Enter** หรือคลิกปุ่ม ▶ แบบฝึกหัด ⭐ **Exercise** แต่ละข้อกำหนดให้แก้ไขเพียงเล็กน้อย เช่น เพิ่มชื่อในรายการ หรือพิมพ์ข้อความหนึ่งบรรทัด ถัดจากแบบฝึกหัดทุกข้อมีเซลล์ ✅ **เฉลย** ผู้เรียนควรทำแบบฝึกหัดด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วทำขั้นตอนถัดไป จากนั้นตอบคำถาม ✍️ ในเซลล์ข้อความ **คำตอบดังกล่าวเป็นชิ้นงานที่ต้องส่ง**
> - 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้:** อยู่ท้ายโน้ตบุ๊ก (notebook) เป็นการทำ OCR กับรายงานผลการตรวจทางห้องปฏิบัติการสังเคราะห์ 20 ฉบับ และรับคะแนนโดยอัตโนมัติ ไม่นับเป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง
>
> ปฏิบัติการนี้ทำงานได้บน [Google Colab](https://colab.research.google.com) รุ่นไม่มีค่าใช้จ่าย ด้วย runtime แบบ CPU ตามปกติ **โดยไม่จำเป็นต้องใช้ GPU** หากพบปัญหา สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom


In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เป็นการ "เตรียมความพร้อม" ก่อนเริ่มปฏิบัติการ โดยทำ 3 อย่าง ได้แก่
#   1) ติดตั้งเครื่องมือ (ไลบรารี) เพิ่มเติมที่จำเป็นต้องใช้ใน Google Colab
#   2) กำหนดค่าพื้นฐาน เช่น รหัสปฏิบัติการ ชื่อที่จะแสดง และรหัสเข้าร่วมตารางอันดับ
#   3) สร้าง "ฟังก์ชัน" (ชุดคำสั่งสำเร็จรูปที่เรียกใช้ซ้ำได้) 2 ตัว คือ
#        - lab_data(...)              : ดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#        - submit_to_leaderboard(...) : ส่งผลการทำนายไปยังตารางอันดับ (ไม่บังคับ)
#   ผู้เรียนเพียงกดรันเซลล์นี้ 1 ครั้ง ไม่จำเป็นต้องแก้ไขโค้ด
#   (ยกเว้นต้องการเข้าร่วมตารางอันดับ ให้กรอก NAME และ JOIN_CODE ด้านล่าง)
# ============================================================================

# ติดตั้งไลบรารีเพิ่มเติมลงในเครื่อง Colab
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - pip install : คำสั่งติดตั้งไลบรารีของ Python
#   - -q (quiet)  : ให้แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - gradio_client   : ไลบรารีสำหรับติดต่อกับเว็บแอปตารางอันดับ (Hugging Face Space)
#   - huggingface_hub : ไลบรารีสำหรับดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face
!pip -q install gradio_client huggingface_hub

# สร้างตัวแปร (ช่องเก็บค่า) HF_OWNER แล้วใส่ข้อความ "wdittaya" ไว้
#   เครื่องหมาย = หมายถึง "นำค่าทางขวา ไปเก็บไว้ในชื่อทางซ้าย"
#   ข้อความในภาษา Python ต้องอยู่ในเครื่องหมายคำพูด " "
HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
# รหัสของปฏิบัติการนี้ ใช้ระบุว่าผลที่ส่งเป็นของปฏิบัติการใด (ไม่ต้องแก้ไข)
LAB_ID    = "lab07"
# ชื่อที่จะแสดงบนตารางอันดับ: พิมพ์ชื่อไว้ระหว่างเครื่องหมาย " " เช่น NAME = "หมอเอ"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
# รหัสเข้าร่วมตารางอันดับ: พิมพ์รหัสที่ผู้สอนแจ้งไว้ระหว่างเครื่องหมาย " "
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

# สร้างชื่อเว็บแอปตารางอันดับ โดยนำ HF_OWNER มาต่อกับข้อความ
#   - f"..." (f-string) คือข้อความที่แทรกค่าตัวแปรได้ โดยเขียนชื่อตัวแปรไว้ใน { }
#   - ผลลัพธ์คือ "wdittaya/aidh-leaderboard"
SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
# สร้างชื่อคลังข้อมูล (dataset) ของรายวิชาในลักษณะเดียวกัน ผลลัพธ์คือ "wdittaya/aidh-lab-data"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

# นำเข้า (import) ฟังก์ชัน hf_hub_download จากไลบรารี huggingface_hub
#   ฟังก์ชันนี้ใช้ดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face มาเก็บไว้ในเครื่อง
from huggingface_hub import hf_hub_download

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน lab_data สำหรับดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#   - def คือคำสั่งสร้างฟังก์ชันใหม่ ตามด้วยชื่อฟังก์ชันและพารามิเตอร์ในวงเล็บ
#   - path : พารามิเตอร์ (ค่าที่ผู้ใช้ส่งเข้ามา) คือชื่อไฟล์ที่ต้องการ เช่น "pima/test_ids.csv"
#   - บรรทัดที่ย่อหน้าเข้าไปด้านล่าง คือคำสั่งที่อยู่ภายในฟังก์ชัน
# ----------------------------------------------------------------------------
def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    # เรียก hf_hub_download(...) เพื่อดาวน์โหลดไฟล์ แล้วส่งตำแหน่งไฟล์ในเครื่องกลับไป (return)
    #   - DATA_REPO           : ชื่อคลังข้อมูลบน Hugging Face ที่เก็บไฟล์
    #   - path                : ชื่อไฟล์ที่ต้องการดาวน์โหลดภายในคลังข้อมูลนั้น
    #   - repo_type="dataset" : ระบุว่าคลังนี้เป็นประเภท "ชุดข้อมูล" (ไม่ใช่โมเดล)
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน submit_to_leaderboard สำหรับส่งไฟล์ผลการทำนายไปยังตารางอันดับ
#   - path   : ชื่อไฟล์ผลการทำนายที่ต้องการส่ง
#   - lab=LAB_ID : รหัสปฏิบัติการ หากไม่ระบุ จะใช้ค่า LAB_ID ที่กำหนดไว้ด้านบนโดยอัตโนมัติ
# ----------------------------------------------------------------------------
def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    # นำเข้าเครื่องมือ 2 ตัวจากไลบรารี gradio_client
    #   - Client      : ใช้เชื่อมต่อกับเว็บแอปตารางอันดับ
    #   - handle_file : ใช้เตรียมไฟล์ในเครื่องให้อยู่ในรูปแบบที่ส่งขึ้นเว็บแอปได้
    from gradio_client import Client, handle_file
    # assert คือการตรวจสอบเงื่อนไข หากเงื่อนไขเป็นเท็จ โปรแกรมจะหยุดและแสดงข้อความเตือน
    #   - NAME and JOIN_CODE : เงื่อนไขว่าต้องกรอกทั้ง NAME และ JOIN_CODE แล้ว (ไม่เป็นข้อความว่าง)
    #   - ข้อความหลังเครื่องหมายจุลภาค คือข้อความเตือนที่จะแสดงเมื่อยังไม่ได้กรอก
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    # เชื่อมต่อกับเว็บแอปตารางอันดับ แล้วเก็บการเชื่อมต่อไว้ในตัวแปร client
    #   - SPACE_ID      : ชื่อเว็บแอปตารางอันดับที่กำหนดไว้ด้านบน
    #   - verbose=False : ไม่ต้องแสดงข้อความรายละเอียดระหว่างเชื่อมต่อ
    client = Client(SPACE_ID, verbose=False)
    # ส่งข้อมูลไปยังเว็บแอปด้วย client.predict(...) แล้วแสดงผลลัพธ์ (คะแนนและอันดับ) ด้วย print(...)
    #   - lab=lab                 : รหัสปฏิบัติการ
    #   - name=NAME               : ชื่อที่จะแสดงบนตารางอันดับ
    #   - code=JOIN_CODE          : รหัสเข้าร่วม
    #   - file=handle_file(path)  : ไฟล์ผลการทำนายที่เตรียมพร้อมสำหรับส่ง
    #   - api_name="/submit"      : ชื่อช่องทางรับข้อมูลของเว็บแอป (ส่วน "ส่งผล")
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

# แสดงข้อความยืนยันว่าการตั้งค่าเสร็จแล้ว ด้วยฟังก์ชัน print(...) (พิมพ์ข้อความออกทางหน้าจอ)
#   - ข้อความเป็น f-string จึงแทรกค่า SPACE_ID, LAB_ID และ NAME ลงในข้อความได้
#   - NAME or '(ยังไม่ได้ระบุ ...)' : หาก NAME ว่าง จะแสดงข้อความในวงเล็บแทน
print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## ความสำคัญของการประมวลผลเอกสาร

ข้อมูลสุขภาพจำนวนมากอยู่ในรูปแบบ PDF ภาพสแกน และข้อความอิสระ (free text) เช่น ใบส่งตัว รายงานผลการตรวจทางห้องปฏิบัติการ และสรุปการจำหน่ายผู้ป่วย (discharge summary) การแปลงข้อมูลเหล่านี้เป็นข้อมูลแบบมีโครงสร้าง ซึ่งคอมพิวเตอร์สามารถเรียงลำดับ ระบุค่าผิดปกติ และนำเสนอเป็นกราฟได้ จึงเป็นงานพื้นฐานที่สำคัญของสุขภาพดิจิทัล

กระบวนการ (pipeline) ในปฏิบัติการนี้ประกอบด้วย 3 ขั้นตอน ดังนี้

1. **OCR** (*Optical Character Recognition* — การรู้จำอักขระด้วยแสง) คือการใช้โปรแกรมอ่านภาพของหน้าเอกสารและแปลงเป็นข้อความ ปฏิบัติการนี้ใช้ **Tesseract** ซึ่งเป็นโปรแกรม OCR แบบโอเพนซอร์ส
2. **การสกัดสารสนเทศ (information extraction)** คือการใช้กฎค้นหาส่วนที่ต้องการในข้อความ ได้แก่ ชื่อการตรวจ ผลการตรวจ หน่วย ชื่อยา และขนาดยา
3. **การตรวจสอบความถูกต้อง** คือการเปรียบเทียบข้อมูลที่สกัดได้กับคำตอบที่ถูกต้อง เพื่อวัดความแม่นยำและตรวจหาการอ่านค่าผิดที่อาจเป็นอันตรายต่อผู้ป่วย

ให้รันเซลล์ถัดไปเพื่อติดตั้ง Tesseract (ใช้เวลาประมาณ 1 นาที)


In [ ]:
# ติดตั้งโปรแกรม OCR Tesseract และไลบรารี Python สำหรับเรียกใช้งาน (ประมาณ 1 นาที; ใช้ runtime แบบ CPU ได้)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เตรียม "เครื่องมือ" ทั้งหมดที่ต้องใช้ในปฏิบัติการการประมวลผลเอกสาร ได้แก่
#   1) ติดตั้งโปรแกรม Tesseract ซึ่งเป็นโปรแกรมการรู้จำอักขระด้วยแสง
#      (OCR — Optical Character Recognition) ทำหน้าที่ "อ่าน" ตัวอักษรจากภาพเอกสาร
#   2) ติดตั้งไลบรารี pytesseract (ตัวเชื่อมให้ Python สั่งงาน Tesseract ได้)
#      และ pillow (ไลบรารีสำหรับเปิด สร้าง และวาดภาพ)
#   3) นำเข้า (import) ไลบรารีต่าง ๆ ที่จะใช้ในเซลล์ถัด ๆ ไป
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ข้อความแสดงเวอร์ชันของ Tesseract ซึ่งยืนยันว่าติดตั้งสำเร็จ
#   ผู้เรียนไม่ต้องแก้ไขโค้ดในเซลล์นี้ เพียงกดรัน 1 ครั้งแล้วรอจนเสร็จ
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: ติดตั้งโปรแกรมและไลบรารี (บรรทัดที่ขึ้นต้นด้วย ! คือคำสั่งระบบปฏิบัติการ)
# ----------------------------------------------------------------------------
# ติดตั้งโปรแกรม Tesseract ลงในเครื่อง Colab (ซึ่งใช้ระบบปฏิบัติการ Linux)
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - apt-get install : คำสั่งติดตั้งโปรแกรมของระบบ Linux (ตระกูล Ubuntu/Debian)
#   - -qq             : ให้แสดงข้อความระหว่างติดตั้งน้อยที่สุด (เงียบกว่า -q)
#   - -y              : ตอบ "ใช่ (yes)" ให้อัตโนมัติเมื่อโปรแกรมถามยืนยันการติดตั้ง
#   - tesseract-ocr   : ชื่อแพ็กเกจของโปรแกรม OCR Tesseract
#   - > /dev/null     : ส่งข้อความที่แสดงออกมาทิ้งไป เพื่อไม่ให้หน้าจอรก
!apt-get -qq install -y tesseract-ocr > /dev/null
# ติดตั้งไลบรารีของ Python ด้วย pip (คำสั่งติดตั้งไลบรารีของ Python)
#   - -q          : แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - pytesseract : ไลบรารีที่ทำให้โค้ด Python เรียกใช้โปรแกรม Tesseract ได้
#   - pillow      : ไลบรารีจัดการภาพ (เมื่อนำเข้าในโค้ดจะใช้ชื่อ PIL)
!pip -q install pytesseract pillow

# ----------------------------------------------------------------------------
# ส่วนที่ 2: นำเข้า (import) ไลบรารีที่จะใช้ในปฏิบัติการ
#   import คือการ "หยิบ" ชุดเครื่องมือสำเร็จรูปมาใช้ในโน้ตบุ๊กนี้
# ----------------------------------------------------------------------------
# นำเข้าไลบรารีมาตรฐานของ Python 2 ตัว (คั่นด้วยเครื่องหมายจุลภาค , เพื่อนำเข้าหลายตัวในบรรทัดเดียว)
#   - re       : ไลบรารี regular expression (นิพจน์ปรกติ) ใช้ค้นหาข้อความตาม "รูปแบบ"
#                เช่น "ชื่อการตรวจ ตามด้วยตัวเลข" (จะอธิบายละเอียดในขั้นตอนที่ 3)
#   - difflib  : ไลบรารีเปรียบเทียบความแตกต่างระหว่างข้อความ 2 ชุด
#                (ใช้เทียบข้อความจาก OCR กับข้อความต้นฉบับ)
import re, difflib
# นำเข้าไลบรารีเสริมอีก 4 ตัว
#   - pytesseract              : ใช้สั่ง Tesseract ให้อ่านข้อความจากภาพ
#   - numpy as np              : ไลบรารีคำนวณตัวเลข; "as np" คือการตั้งชื่อย่อให้เรียกสั้น ๆ ว่า np
#   - pandas as pd             : ไลบรารีจัดการข้อมูลแบบตาราง (DataFrame) ตั้งชื่อย่อว่า pd
#   - matplotlib.pyplot as plt : ไลบรารีวาดกราฟและแสดงภาพ ตั้งชื่อย่อว่า plt
import pytesseract, numpy as np, pandas as pd, matplotlib.pyplot as plt
# นำเข้าเครื่องมือ 3 ตัวจากไลบรารี PIL (ชื่อที่ใช้เรียกไลบรารี pillow)
#   - from ... import ... : หยิบเฉพาะเครื่องมือที่ต้องการออกมาจากไลบรารี
#   - Image     : ใช้สร้าง เปิด และบันทึกภาพ
#   - ImageDraw : ใช้วาดรูปหรือเขียนข้อความลงบนภาพ
#   - ImageFont : ใช้โหลดแบบอักษร (ฟอนต์) สำหรับเขียนข้อความลงบนภาพ
from PIL import Image, ImageDraw, ImageFont
# แสดงเวอร์ชันของโปรแกรม Tesseract ทางหน้าจอ เพื่อยืนยันว่าติดตั้งสำเร็จ
#   - print(...) : ฟังก์ชันพิมพ์ข้อความออกทางหน้าจอ; ใส่ได้หลายค่าคั่นด้วย , (จะเว้นวรรคให้อัตโนมัติ)
#   - "เวอร์ชันของ Tesseract:" : ข้อความนำหน้า (อยู่ในเครื่องหมายคำพูด " ")
#   - pytesseract.get_tesseract_version() : ฟังก์ชันของ pytesseract ที่ถามเลขเวอร์ชันจากโปรแกรม Tesseract
#     (วงเล็บว่าง () หมายถึงเรียกใช้ฟังก์ชันโดยไม่ต้องส่งค่าใดเข้าไป)
print("เวอร์ชันของ Tesseract:", pytesseract.get_tesseract_version())

## ขั้นตอนที่ 1 — รายงานผลการตรวจทางห้องปฏิบัติการที่ผ่านการสแกน (ข้อมูลสังเคราะห์)

เซลล์ถัดไปสร้างภาพรายงานผลการตรวจทางห้องปฏิบัติการ เพื่อให้ผู้เรียนทุกคนใช้ข้อมูลนำเข้าชุดเดียวกัน ผู้ป่วยในรายงานเป็นบุคคลสมมติ **ห้ามใช้เอกสารของผู้ป่วยจริง (ข้อมูลสุขภาพส่วนบุคคล หรือ PHI) ในโน้ตบุ๊กสาธารณะ** ข้อความในรายงานเป็นภาษาอังกฤษตามรูปแบบของรายงานผลการตรวจจริง


In [ ]:
# สร้างภาพ 'รายงานผลการตรวจทางห้องปฏิบัติการ' สังเคราะห์ (ไม่มีข้อมูลผู้ป่วยจริง)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้จำลอง "ภาพสแกนรายงานผลการตรวจทางห้องปฏิบัติการ" ขึ้นมา 1 ภาพ
#   เพื่อให้ผู้เรียนทุกคนใช้เอกสารต้นฉบับชุดเดียวกัน และไม่ต้องใช้เอกสารของผู้ป่วยจริง
#   (ข้อมูลสุขภาพส่วนบุคคล หรือ PHI — Protected Health Information ห้ามนำมาใช้ในโน้ตบุ๊กสาธารณะ)
#   ขั้นตอนการทำงาน:
#     1) เขียนเนื้อหารายงาน (ชื่อการตรวจ ผล หน่วย ช่วงค่าอ้างอิง รายการยา) เป็นข้อความ
#     2) สร้างภาพพื้นขาวว่าง ๆ แล้ว "พิมพ์" ข้อความนั้นลงบนภาพด้วยตัวอักษรสีดำ
#     3) บันทึกภาพเป็นไฟล์ report.png (ไฟล์นี้จะถูกนำไปทำ OCR ในขั้นตอนถัดไป)
#     4) แสดงภาพบนหน้าจอให้ผู้เรียนดู
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ภาพรายงานผลแล็บของผู้ป่วยสมมติ (John Doe)
#   ข้อสังเกต: สำหรับคอมพิวเตอร์ ภาพนี้เป็นเพียงจุดสี (pixel) ยังค้นหาคำในภาพไม่ได้
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: เนื้อหาของรายงาน (เก็บเป็นข้อความ)
# ----------------------------------------------------------------------------
# สร้างตัวแปร report_text เก็บเนื้อหารายงานทั้งฉบับ
#   - ตัวแปร คือ "ช่องเก็บค่า" ที่ตั้งชื่อไว้เรียกใช้ภายหลัง; เครื่องหมาย = หมายถึงนำค่าทางขวาไปเก็บไว้ในชื่อทางซ้าย
#   - เครื่องหมายคำพูดสามตัว """ ... """ ใช้สร้างข้อความที่ยาวหลายบรรทัด (ขึ้นบรรทัดใหม่ได้ตามที่พิมพ์)
#   - เนื้อหาประกอบด้วย: หัวรายงาน, ข้อมูลผู้ป่วยสมมติ (ชื่อ, MRN = เลขเวชระเบียน, อายุ, เพศ), วันที่,
#     ตารางผลการตรวจ 4 รายการ (TEST = ชื่อการตรวจ, RESULT = ผล, UNIT = หน่วย, REFERENCE = ช่วงค่าอ้างอิง),
#     รายการยา (Medications) โดย BID = วันละ 2 ครั้ง, OD = วันละ 1 ครั้ง
#     และความเห็นสรุป (Impression): ภาวะโลหิตจาง น้ำตาลในเลือดสูง และ LDL สูง
report_text = """CENTRAL HOSPITAL - LABORATORY REPORT
Patient: John Doe    MRN: 001234    Age: 57    Sex: M
Date: 2026-10-17

TEST                RESULT     UNIT       REFERENCE
Hemoglobin          11.2       g/dL       13.5-17.5
Glucose (fasting)   142        mg/dL      70-99
Creatinine          1.8        mg/dL      0.7-1.3
LDL cholesterol     165        mg/dL      <100

Medications: Metformin 500 mg BID; Atorvastatin 20 mg OD
Impression: Anemia, hyperglycemia, elevated LDL."""

# ----------------------------------------------------------------------------
# ส่วนที่ 2: เลือกแบบอักษร (ฟอนต์) สำหรับเขียนข้อความลงบนภาพ
#   ใช้โครงสร้าง try / except เพื่อ "ลองทำ" และ "มีทางสำรอง" หากเกิดข้อผิดพลาด
# ----------------------------------------------------------------------------
# try: คือ "ลองทำคำสั่งที่ย่อหน้าอยู่ด้านล่าง" หากเกิดข้อผิดพลาด โปรแกรมจะไม่หยุด
#   แต่จะกระโดดไปทำคำสั่งในส่วน except แทน
try:
    # โหลดฟอนต์ DejaVu Sans Mono จากไฟล์ในเครื่อง แล้วเก็บไว้ในตัวแปร font
    #   - ImageFont.truetype(...) : ฟังก์ชันของ PIL สำหรับโหลดฟอนต์ชนิด TrueType (.ttf)
    #   - "/usr/share/fonts/.../DejaVuSansMono.ttf" : ตำแหน่งไฟล์ฟอนต์ในเครื่อง Colab
    #     (ฟอนต์ Mono คือฟอนต์ที่ทุกตัวอักษรกว้างเท่ากัน ทำให้คอลัมน์ของตารางตรงกัน)
    #   - 20 : ขนาดตัวอักษร (หน่วยเป็นจุดภาพ pixel)
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", 20)   # มีอยู่ใน Colab
# except OSError: คือ "หากเกิดข้อผิดพลาดประเภท OSError" (เช่น หาไฟล์ฟอนต์ไม่พบ) ให้ทำคำสั่งด้านล่างแทน
except OSError:
    # ใช้ฟอนต์พื้นฐานที่มากับไลบรารี PIL แทน (ทางสำรองเมื่อไม่มีไฟล์ฟอนต์ด้านบน)
    #   - ImageFont.load_default() : โหลดฟอนต์มาตรฐานของ PIL โดยไม่ต้องระบุไฟล์
    font = ImageFont.load_default()

# ----------------------------------------------------------------------------
# ส่วนที่ 3: สร้างภาพและเขียนข้อความลงบนภาพ แล้วบันทึกเป็นไฟล์
# ----------------------------------------------------------------------------
# สร้างภาพใหม่ที่เป็นพื้นสีขาวทั้งภาพ แล้วเก็บไว้ในตัวแปร img
#   - Image.new(...) : ฟังก์ชันของ PIL สำหรับสร้างภาพเปล่า
#   - "RGB"          : ชนิดของภาพเป็นภาพสี (แดง-เขียว-น้ำเงิน)
#   - (900, 520)     : ขนาดภาพ กว้าง 900 จุดภาพ x สูง 520 จุดภาพ
#                      (วงเล็บ ( ) ที่มีค่าคั่นด้วย , เรียกว่า tuple คือกลุ่มของค่าที่ใช้คู่กัน)
#   - "white"        : สีพื้นหลังของภาพเป็นสีขาว
img = Image.new("RGB", (900, 520), "white")
# สร้าง "ปากกา" สำหรับวาดหรือเขียนลงบนภาพ img แล้วเก็บไว้ในตัวแปร d
#   - ImageDraw.Draw(img) : สร้างเครื่องมือวาดที่ผูกกับภาพ img (สิ่งที่วาดจะปรากฏบนภาพนี้)
d = ImageDraw.Draw(img)
# เขียนข้อความหลายบรรทัด (report_text) ลงบนภาพ
#   - d.multiline_text(...) : คำสั่งเขียนข้อความที่มีหลายบรรทัดลงบนภาพ
#   - (20, 20)       : ตำแหน่งเริ่มเขียน (ห่างจากขอบซ้าย 20 จุดภาพ, ห่างจากขอบบน 20 จุดภาพ)
#   - report_text    : ข้อความที่จะเขียน
#   - fill="black"   : สีของตัวอักษรเป็นสีดำ
#   - spacing=8      : ระยะห่างระหว่างบรรทัด 8 จุดภาพ
#   - font=font      : ใช้ฟอนต์ที่เลือกไว้ในส่วนที่ 2
d.multiline_text((20, 20), report_text, fill="black", spacing=8, font=font)
# บันทึกภาพเป็นไฟล์ชื่อ report.png ในเครื่อง Colab
#   - img.save(...)  : คำสั่งบันทึกภาพลงไฟล์ (ชนิดไฟล์ดูจากนามสกุล .png)
img.save("report.png")

# ----------------------------------------------------------------------------
# ส่วนที่ 4: แสดงภาพบนหน้าจอ
# ----------------------------------------------------------------------------
# บรรทัดนี้มี 4 คำสั่งคั่นด้วยเครื่องหมาย ; (เซมิโคลอน ใช้เขียนหลายคำสั่งในบรรทัดเดียว) ทั้งหมดมาจาก matplotlib (plt)
#   1) plt.figure(figsize=(9, 6)) : สร้างพื้นที่แสดงภาพ ขนาดกว้าง 9 นิ้ว x สูง 6 นิ้ว
#   2) plt.imshow(img)            : นำภาพ img มาแสดงในพื้นที่นั้น
#   3) plt.axis("off")            : ซ่อนแกนและตัวเลขพิกัดรอบภาพ ให้เห็นเฉพาะตัวภาพ
#   4) plt.show()                 : สั่งแสดงผลภาพออกทางหน้าจอ
plt.figure(figsize=(9, 6)); plt.imshow(img); plt.axis("off"); plt.show()

## ขั้นตอนที่ 2 — การแปลงภาพเป็นข้อความด้วย OCR

สำหรับคอมพิวเตอร์ ไฟล์ `report.png` เป็นเพียงตารางของจุดสี (pixel) ระบบจึงไม่สามารถค้นหาคำว่า "Creatinine" ในภาพได้โดยตรง OCR ทำหน้าที่รู้จำตัวอักษรและแปลงภาพเป็นข้อความธรรมดา


In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: การทำ OCR กับภาพ
# ให้แทนที่ ... ด้วยโค้ดบรรทัดต่อไปนี้ (สามารถคัดลอกและวางได้):
#     pytesseract.image_to_string(Image.open("report.png"), config="--psm 6")
# คำแนะนำ: Image.open(...) ใช้เปิดไฟล์ภาพ ส่วน image_to_string(...) ใช้อ่านข้อความจากภาพ
#          "--psm 6" กำหนดให้ Tesseract ถือว่าหน้าเอกสารเป็นข้อความหลายบรรทัดในกลุ่มเดียว (เหมาะกับตาราง)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ใช้ OCR (การรู้จำอักขระด้วยแสง) "อ่าน" ภาพ report.png ที่สร้างไว้ในขั้นตอนที่ 1
#   ให้กลายเป็นข้อความธรรมดาที่คอมพิวเตอร์ค้นหาและประมวลผลต่อได้
#   ข้อมูลนำเข้า : ไฟล์ภาพ report.png
#   ผลลัพธ์      : ข้อความที่ Tesseract อ่านได้ เก็บไว้ในตัวแปร ocr_text และแสดงบนหน้าจอ
#   สิ่งที่ผู้เรียนต้องทำ: แทนที่ ... ด้วยโค้ดตามที่ระบุไว้ด้านบน แล้วรันเซลล์
# ============================================================================

# สร้างตัวแปร ocr_text เพื่อเก็บข้อความที่ได้จากการทำ OCR
#   - เครื่องหมาย ... (จุดสามจุด) คือ "ช่องว่าง" ที่ผู้เรียนต้องเติมโค้ดเอง
#     (หากรันโดยยังไม่แก้ไข ocr_text จะเก็บค่า ... ไว้ และหน้าจอจะแสดงคำว่า Ellipsis)
#   - ให้แทนที่ ... ด้วยโค้ดที่ให้ไว้ในคำแนะนำด้านบน ซึ่งทำงาน 2 ขั้นจากในออกนอก:
#       1) Image.open("report.png") : เปิดไฟล์ภาพด้วยไลบรารี PIL
#       2) pytesseract.image_to_string(ภาพ, config="--psm 6") : ส่งภาพให้ Tesseract อ่านเป็นข้อความ
#          - config : ตัวเลือกเพิ่มเติมที่ส่งให้ Tesseract
#          - "--psm 6" : โหมดการแบ่งหน้า (page segmentation mode) แบบที่ 6
#                        = มองทั้งภาพเป็นข้อความกลุ่มเดียวที่มีหลายบรรทัด
#   - # TODO ท้ายบรรทัด คือป้ายเตือนว่าบรรทัดนี้ยังต้องแก้ไข
ocr_text = ...    # TODO
# แสดงข้อความที่ได้จาก OCR ออกทางหน้าจอด้วย print(...) เพื่อให้ผู้เรียนเทียบกับภาพต้นฉบับ
print(ocr_text)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 1)
#   ทำ OCR กับภาพ report.png เพื่อแปลงภาพรายงานผลแล็บให้เป็นข้อความธรรมดา
#   ข้อมูลนำเข้า : ไฟล์ภาพ report.png (สร้างไว้ในขั้นตอนที่ 1)
#   ผลลัพธ์      : ตัวแปร ocr_text ที่เก็บข้อความจาก OCR และข้อความนั้นถูกแสดงบนหน้าจอ
#   ข้อสังเกต: OCR ให้ผลเป็น "ข้อความ" มิใช่ตาราง คอลัมน์ต่าง ๆ จะถูกคั่นด้วยช่องว่าง
# ============================================================================

# อ่านข้อความจากภาพด้วย OCR แล้วเก็บไว้ในตัวแปร ocr_text (ทำงาน 2 ขั้น จากวงเล็บในออกนอก)
#   1) Image.open("report.png") : ฟังก์ชันของไลบรารี PIL เปิดไฟล์ภาพ report.png ขึ้นมาเป็นภาพในหน่วยความจำ
#   2) pytesseract.image_to_string(...) : ฟังก์ชันของไลบรารี pytesseract ส่งภาพให้โปรแกรม Tesseract
#      อ่านตัวอักษร แล้วคืนค่าเป็นข้อความ (string)
#      - อาร์กิวเมนต์แรก : ภาพที่ต้องการอ่าน (ผลจากขั้นที่ 1)
#      - config="--psm 6" : ตัวเลือกส่งให้ Tesseract; psm (page segmentation mode) โหมด 6
#                           = ถือว่าทั้งภาพเป็นข้อความกลุ่มเดียวหลายบรรทัด เหมาะกับรายงานแบบตาราง
ocr_text = pytesseract.image_to_string(Image.open("report.png"), config="--psm 6")
# แสดงข้อความที่ OCR อ่านได้ออกทางหน้าจอ
print(ocr_text)

In [ ]:
# เปรียบเทียบข้อความจาก OCR กับข้อความต้นฉบับในภาพ (ไม่นับช่องว่างส่วนเกิน)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ตรวจสอบว่า OCR อ่านภาพรายงานได้ถูกต้องเพียงใด โดยเทียบข้อความ 2 ชุดทีละบรรทัด
#     - report_text : ข้อความต้นฉบับที่ใช้สร้างภาพ (คำตอบที่ถูกต้อง)
#     - ocr_text    : ข้อความที่ OCR อ่านได้จากภาพ
#   ก่อนเทียบจะ "ทำความสะอาด" ข้อความ คือรวบช่องว่างหลายช่องให้เหลือช่องเดียวและตัดบรรทัดว่างทิ้ง
#   เพราะ OCR มักแทนช่องว่างระหว่างคอลัมน์ด้วยช่องว่างเพียงช่องเดียว ซึ่งไม่ถือว่าอ่านผิด
#   ผลลัพธ์ที่ผู้เรียนจะเห็น:
#     - จำนวนบรรทัดของต้นฉบับและของ OCR
#     - บรรทัดที่ไม่ตรงกัน (ขึ้นต้นด้วย - คือในเอกสาร, + คือที่ OCR อ่านได้)
#       หรือข้อความ ✅ หากตรงกันทุกตัวอักษร
#   ผลนี้ใช้ตอบคำถามที่ 1 เรื่องความผิดพลาดของ OCR
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: สร้างฟังก์ชัน squash สำหรับทำความสะอาดข้อความ
#   - def คือคำสั่งสร้างฟังก์ชัน (ชุดคำสั่งที่ตั้งชื่อไว้และเรียกใช้ซ้ำได้)
#   - s : พารามิเตอร์ คือข้อความที่ส่งเข้ามาให้ฟังก์ชันทำความสะอาด
#   - ผลลัพธ์ (return) : รายการ (list) ของบรรทัดที่ไม่ว่าง โดยแต่ละบรรทัดมีช่องว่างคั่นคำเพียงช่องเดียว
# ----------------------------------------------------------------------------
def squash(s):
    # ส่งรายการบรรทัดที่ทำความสะอาดแล้วกลับไป (return = ส่งผลลัพธ์ออกจากฟังก์ชัน)
    #   บรรทัดนี้เขียนแบบ list comprehension คือการสร้างรายการใหม่ด้วยรูปแบบ
    #   [ สิ่งที่จะใส่ในรายการ  for ตัวแปร in แหล่งข้อมูล  if เงื่อนไข ] อ่านจากกลางไปซ้ายได้ดังนี้
    #   1) s.splitlines()         : ตัดข้อความ s ออกเป็นรายการของบรรทัด (ตัดตรงจุดขึ้นบรรทัดใหม่)
    #   2) for line in ...        : วนหยิบทีละบรรทัดมาเก็บในตัวแปร line
    #   3) if line.strip()        : เก็บเฉพาะบรรทัดที่ไม่ว่าง
    #                               (.strip() ตัดช่องว่างหัวท้าย; ถ้าเหลือข้อความว่าง ถือว่าเป็นเท็จ จึงถูกข้าม)
    #   4) line.split()           : แยกบรรทัดเป็นคำ ๆ โดยตัดตรงช่องว่าง (ช่องว่างกี่ช่องก็ถือเป็นตัวคั่นเดียว)
    #   5) " ".join(...)          : นำคำเหล่านั้นมาต่อกันใหม่ โดยคั่นด้วยช่องว่าง 1 ช่อง
    return [" ".join(line.split()) for line in s.splitlines() if line.strip()]

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ทำความสะอาดข้อความทั้งสองชุด แล้วแสดงจำนวนบรรทัด
# ----------------------------------------------------------------------------
# เรียกฟังก์ชัน squash กับข้อความ 2 ชุด แล้วเก็บผลลงตัวแปร 2 ตัวพร้อมกันในบรรทัดเดียว
#   - ฝั่งซ้ายและขวาของ = มี 2 ค่าคั่นด้วย , ค่าแรกไปเก็บในตัวแปรแรก ค่าที่สองไปเก็บในตัวแปรที่สอง
#   - original  ← squash(report_text) : บรรทัดของข้อความต้นฉบับ
#   - ocr_lines ← squash(ocr_text)    : บรรทัดของข้อความที่ OCR อ่านได้
original, ocr_lines = squash(report_text), squash(ocr_text)
# แสดงจำนวนบรรทัดของทั้งสองชุด
#   - f"..." (f-string) : ข้อความที่แทรกค่าได้ โดยเขียนคำสั่งไว้ใน { }
#   - len(original)     : ฟังก์ชัน len นับจำนวนสมาชิกในรายการ (ที่นี่คือจำนวนบรรทัด)
#   - \n                : สัญลักษณ์ขึ้นบรรทัดใหม่ (ทำให้มีบรรทัดว่างคั่นก่อนผลลัพธ์ถัดไป)
print(f"ต้นฉบับ {len(original)} บรรทัด, OCR อ่านได้ {len(ocr_lines)} บรรทัด\n")

# ----------------------------------------------------------------------------
# ส่วนที่ 3: หาบรรทัดที่แตกต่างกัน แล้วแสดงออกมา พร้อมนับจำนวน
# ----------------------------------------------------------------------------
# สร้างตัวแปร n_diff เป็นตัวนับจำนวนบรรทัดที่แตกต่าง เริ่มต้นที่ 0
n_diff = 0
# วนลูป for เพื่อหยิบผลการเปรียบเทียบออกมาทีละบรรทัด (คำสั่งที่ย่อหน้าด้านล่างจะทำซ้ำกับทุกบรรทัด)
#   - difflib.unified_diff(...) : ฟังก์ชันของไลบรารี difflib เปรียบเทียบรายการบรรทัด 2 ชุด
#     แล้วให้ผลในรูปแบบ "unified diff" (บรรทัดขึ้นต้น - = มีเฉพาะในชุดแรก, + = มีเฉพาะในชุดที่สอง)
#     - original   : ชุดข้อมูลแรก (ต้นฉบับ)
#     - ocr_lines  : ชุดข้อมูลที่สอง (ผล OCR)
#     - "typed"    : ชื่อเรียกชุดแรก ใช้ในบรรทัดหัวเรื่อง (บรรทัดที่ขึ้นต้นด้วย ---)
#     - "OCR"      : ชื่อเรียกชุดที่สอง ใช้ในบรรทัดหัวเรื่อง (บรรทัดที่ขึ้นต้นด้วย +++)
#     - lineterm="" : ไม่ต้องเติมสัญลักษณ์ขึ้นบรรทัดใหม่ท้ายบรรทัดหัวเรื่อง (เพราะ print จะขึ้นบรรทัดให้อยู่แล้ว)
#     - n=0        : ไม่ต้องแสดงบรรทัดที่เหมือนกันรอบ ๆ จุดที่ต่าง (แสดงเฉพาะส่วนที่ต่างจริง)
#   - line : ตัวแปรที่รับผลการเปรียบเทียบทีละบรรทัดในแต่ละรอบ
for line in difflib.unified_diff(original, ocr_lines, "typed", "OCR", lineterm="", n=0):
    # ตรวจเงื่อนไขด้วย if: เลือกเฉพาะบรรทัดที่เป็น "ความแตกต่างจริง" (ไม่ใช่บรรทัดหัวเรื่อง)
    #   - line.startswith(("-", "+"))  : บรรทัดขึ้นต้นด้วย - หรือ + หรือไม่
    #                                    (ใส่หลายค่าใน ( ) เพื่อถามว่าขึ้นต้นด้วยค่าใดค่าหนึ่งหรือไม่)
    #   - and not line.startswith(("---", "+++")) : และต้องไม่ใช่บรรทัดหัวเรื่อง --- typed / +++ OCR
    #   - and = "และ" (ต้องจริงทั้งสองเงื่อนไข), not = "ไม่" (กลับค่าจริงเป็นเท็จ)
    if line.startswith(("-", "+")) and not line.startswith(("---", "+++")):
        # 2 คำสั่งคั่นด้วย ; ทำเมื่อเงื่อนไขเป็นจริง
        #   1) print(line)  : แสดงบรรทัดที่แตกต่างออกทางหน้าจอ
        #   2) n_diff += 1  : เพิ่มค่าตัวนับขึ้น 1 (ย่อมาจาก n_diff = n_diff + 1)
        print(line); n_diff += 1
# แสดงข้อความสรุปผล โดยเลือกข้อความด้วยรูปแบบ "ค่า_A if เงื่อนไข else ค่า_B" (เลือก A เมื่อเงื่อนไขจริง มิฉะนั้นเลือก B)
#   - n_diff == 0 : ไม่มีบรรทัดที่แตกต่างเลย (== คือการเปรียบเทียบว่าเท่ากันหรือไม่)
#   - ถ้าจริง : แสดง "✅ ข้อความจาก OCR ตรงกับต้นฉบับทุกตัวอักษร"
#   - ถ้าเท็จ : แสดงคำอธิบายสัญลักษณ์ - และ + (มี \n ขึ้นบรรทัดใหม่ก่อน)
print("✅ ข้อความจาก OCR ตรงกับต้นฉบับทุกตัวอักษร" if n_diff == 0 else "\n(- = ข้อความในเอกสาร, + = ข้อความที่ OCR อ่านได้)")

💡 **คำอธิบายผลลัพธ์**

Tesseract แปลงภาพเป็นข้อความ โดยคอลัมน์ของตารางถูกแทนด้วยช่องว่างเพียงหนึ่งช่อง กล่าวคือ OCR ให้ผลลัพธ์เป็น *ข้อความ* มิใช่ตาราง สำหรับภาพที่สร้างด้วยคอมพิวเตอร์และมีความคมชัดเช่นนี้ OCR มักอ่านได้ถูกต้องทั้งหมด แต่ภาพสแกนจริง (ภาพเอียง แฟกซ์ มีตราประทับ หรือลายมือ) มักเกิดข้อผิดพลาด เช่น `O`↔`0`, `l`↔`1`, `B`↔`8` ผลกระทบของข้อผิดพลาดดังกล่าวจะแสดงในขั้นตอนที่ 6


✍️ **คำถามที่ 1** จากผลการเปรียบเทียบข้างต้น OCR อ่านรายงานฉบับนี้ผิดพลาดหรือไม่ อย่างไร และโปรดระบุ **ลักษณะของเอกสารทางคลินิกในหน่วยงานของผู้เรียนอย่างน้อย 3 ประการ** ที่ทำให้การทำ OCR ยากขึ้น (เช่น ลายมือ ตราประทับ แฟกซ์ ตาราง หรือข้อความภาษาไทยปนภาษาอังกฤษ)


✍️ **คำตอบ:**



## ขั้นตอนที่ 3 — การสกัดผลการตรวจด้วยรูปแบบการค้นหาสำเร็จรูป

**Regular expression** (*regex* — นิพจน์ปรกติ) คือรูปแบบการค้นหาข้อความ ซึ่งคล้ายกับคำสั่ง "Find" ในโปรแกรม Word แต่สามารถระบุ *ประเภท* ของข้อความได้ (เช่น "ตัวเลขใด ๆ") นอกเหนือจากคำที่ตรงกันทุกตัวอักษร ในปฏิบัติการนี้ผู้เรียน **ไม่จำเป็นต้อง** เขียน regex เอง ความหมายของรูปแบบที่ใช้มีดังนี้

| ส่วนของรูปแบบ | ความหมาย |
|---|---|
| `^` | ที่ตำแหน่งต้นบรรทัด … |
| `(Hemoglobin\|Creatinine\|…)` | … มีชื่อการตรวจชื่อใดชื่อหนึ่งในรายการที่กำหนด (`\|` หมายถึง "หรือ") … |
| `\s+` | … ตามด้วยช่องว่างอย่างน้อย 1 ช่อง … |
| `(\d+(?:\.\d+)?)` | … ตามด้วยตัวเลข เช่น `142` หรือ `1.8` (**ผลการตรวจ**) … |
| `\s+(\S+)` | … ตามด้วยช่องว่างและกลุ่มอักขระที่ไม่มีช่องว่าง เช่น `mg/dL` (**หน่วย**) |

ส่วนที่อยู่ในวงเล็บ `( )` คือส่วนที่ระบบเก็บไว้ ตัวอย่างเช่น บรรทัด `Creatinine 1.8 mg/dL 0.7-1.3` จะได้ผลลัพธ์ `("Creatinine", "1.8", "mg/dL")`
เนื่องจากรูปแบบนี้ค้นหาเฉพาะชื่อที่อยู่ใน **รายการที่กำหนด** การตรวจที่ไม่อยู่ในรายการจะถูกละเว้นโดยไม่มีการแจ้งเตือน ซึ่งเป็นข้อผิดพลาดที่พบบ่อยในการใช้งานจริง


In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: การตรวจสอบว่าสกัดผลการตรวจได้ครบทุกรายการ
# ให้รันเซลล์นี้ แล้วเปรียบเทียบตารางผลลัพธ์กับภาพรายงานในขั้นตอนที่ 1
# ผลการตรวจขาดไป 1 รายการ เนื่องจากชื่อการตรวจนั้นไม่อยู่ในรายการด้านล่าง
# ➜ ให้เพิ่มชื่อการตรวจที่ขาดไปใน KNOWN_TESTS (สะกดให้ตรงกับรายงาน อยู่ภายในเครื่องหมายคำพูด "..."
#   และคั่นด้วยเครื่องหมายจุลภาค ,) จากนั้นรันเซลล์อีกครั้ง ผลลัพธ์ที่ถูกต้องมี 4 แถว
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เปลี่ยน "ข้อความ" ที่ได้จาก OCR ให้เป็น "ตาราง" ผลการตรวจทางห้องปฏิบัติการ
#   (การสกัดสารสนเทศ — information extraction) โดยใช้รูปแบบการค้นหาข้อความ
#   (regular expression หรือ regex — นิพจน์ปรกติ) ค้นหาบรรทัดที่มีลักษณะ "ชื่อการตรวจ  ผล  หน่วย"
#   ข้อมูลนำเข้า : ocr_text (ข้อความจาก OCR) และรายชื่อการตรวจที่รู้จัก KNOWN_TESTS
#   ผลลัพธ์      : ตาราง labs มีคอลัมน์ test (ชื่อการตรวจ), result (ผล), unit (หน่วย)
#   ประเด็นสำคัญทางคลินิก: รูปแบบนี้ค้นหาเฉพาะชื่อที่อยู่ในรายการ หากการตรวจใดไม่อยู่ในรายการ
#   จะ "หายไปเงียบ ๆ" โดยไม่มีการแจ้งเตือน ซึ่งอาจทำให้ผลผิดปกติตกหล่นได้
#   สิ่งที่ผู้เรียนต้องทำ: เทียบตารางกับภาพรายงาน หาชื่อการตรวจที่ขาด แล้วเพิ่มใน KNOWN_TESTS
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: รายชื่อการตรวจที่ต้องการค้นหา
# ----------------------------------------------------------------------------
# สร้างตัวแปร KNOWN_TESTS เก็บ "รายการ" (list) ชื่อการตรวจที่ระบบรู้จัก
#   - list คือกลุ่มของค่าหลายค่าเรียงกัน เขียนไว้ใน [ ] และคั่นแต่ละค่าด้วยเครื่องหมาย ,
#   - แต่ละชื่อเป็นข้อความ จึงต้องอยู่ในเครื่องหมายคำพูด " "
#   - ปัจจุบันมี 3 ชื่อ แต่ภาพรายงานมีการตรวจ 4 รายการ (ยังขาดอีก 1 ชื่อ)
#   - # TODO ท้ายบรรทัด คือจุดที่ผู้เรียนต้องเพิ่มชื่อการตรวจที่ขาดไป
#     (ดูชื่อจากคอลัมน์ TEST ในภาพรายงาน และสะกดให้ตรงทุกตัวอักษร รวมถึงช่องว่าง)
KNOWN_TESTS = ["Hemoglobin", "Glucose (fasting)", "Creatinine"]   # TODO เพิ่มชื่อการตรวจที่ขาดไป

# ----------------------------------------------------------------------------
# ส่วนที่ 2: สร้างฟังก์ชัน extract_labs สำหรับสกัดผลการตรวจจากข้อความ
#   - def คือคำสั่งสร้างฟังก์ชัน (ชุดคำสั่งที่ตั้งชื่อไว้และเรียกใช้ซ้ำได้)
#   - text : พารามิเตอร์ คือข้อความ (เช่น ผลจาก OCR) ที่ต้องการค้นหาผลการตรวจ
#   - ผลลัพธ์ (return) : ตาราง (DataFrame) ที่มี 3 คอลัมน์ คือ test (ชื่อการตรวจ),
#     result (ผลการตรวจเป็นตัวเลข) และ unit (หน่วย)
# ----------------------------------------------------------------------------
def extract_labs(text):
    """ค้นหา 'ชื่อการตรวจ  ผล  หน่วย' ในแต่ละบรรทัดของข้อความ สำหรับทุกชื่อใน KNOWN_TESTS"""
    # (ข้อความในเครื่องหมาย """ ด้านบนเรียกว่า docstring คือคำอธิบายสั้น ๆ ว่าฟังก์ชันนี้ทำอะไร)
    # นำชื่อการตรวจทุกชื่อใน KNOWN_TESTS มาต่อกันเป็นข้อความเดียว คั่นด้วย | แล้วเก็บในตัวแปร names
    #   เช่น ได้ผลประมาณ "Hemoglobin|Glucose\ \(fasting\)|Creatinine"
    #   - for n in KNOWN_TESTS : วนหยิบชื่อการตรวจทีละชื่อมาเก็บในตัวแปร n
    #   - re.escape(n)         : ฟังก์ชันของไลบรารี re ที่ "ใส่เครื่องหมาย \ หน้าสัญลักษณ์พิเศษ"
    #                            เช่น วงเล็บใน "Glucose (fasting)" เพื่อให้ regex ถือเป็นตัวอักษรธรรมดา
    #                            (ปกติ ( ) ใน regex มีความหมายพิเศษว่า "กลุ่มที่ต้องการเก็บ")
    #   - "|".join(...)        : นำทุกชื่อมาต่อกัน โดยใช้ | คั่นกลาง; ใน regex เครื่องหมาย | หมายถึง "หรือ"
    names = "|".join(re.escape(n) for n in KNOWN_TESTS)            # เชื่อมชื่อการตรวจด้วย | (หมายถึง "หรือ")
    # สร้าง "รูปแบบการค้นหา" (regular expression) แล้วเก็บไว้ในตัวแปร pattern
    #   - re.compile(รูปแบบ, ตัวเลือก) : ฟังก์ชันของไลบรารี re ที่แปลงข้อความรูปแบบให้พร้อมใช้ค้นหาซ้ำได้
    #   - rf"..." : r = raw string (ให้ \ เป็นตัวอักษรธรรมดา ไม่ใช่สัญลักษณ์พิเศษของ Python)
    #               f = f-string (แทรกค่าตัวแปร names ลงในตำแหน่ง {names})
    #   ความหมายของรูปแบบทีละส่วน (อ่านจากซ้ายไปขวา):
    #     ^                : ต้องเริ่มที่ต้นบรรทัด
    #     \s*              : ตามด้วยช่องว่าง 0 ช่องขึ้นไป (\s = ช่องว่าง/แท็บ, * = มีกี่ตัวก็ได้รวมถึงไม่มี)
    #     ({names})        : กลุ่มที่ 1 = ชื่อการตรวจชื่อใดชื่อหนึ่งในรายการ (วงเล็บ ( ) = ส่วนที่ต้องการเก็บไว้)
    #     \s+              : ตามด้วยช่องว่างอย่างน้อย 1 ช่อง (+ = 1 ตัวขึ้นไป)
    #     (\d+(?:\.\d+)?)  : กลุ่มที่ 2 = ผลการตรวจที่เป็นตัวเลข
    #                          \d+       = ตัวเลข 0-9 อย่างน้อย 1 หลัก เช่น 142
    #                          (?:\.\d+) = กลุ่ม "จุดทศนิยมตามด้วยตัวเลข" เช่น .8
    #                                      (?: ) คือการจัดกลุ่มโดยไม่เก็บแยก; \. คือจุดจริง ๆ
    #                          ?         = กลุ่มทศนิยมนี้จะมีหรือไม่มีก็ได้ (จึงรับทั้ง 142 และ 1.8)
    #     \s+              : ตามด้วยช่องว่างอย่างน้อย 1 ช่อง
    #     (\S+)            : กลุ่มที่ 3 = หน่วย คืออักขระที่ไม่ใช่ช่องว่างติดกันอย่างน้อย 1 ตัว เช่น mg/dL
    #                        (\S ตัวใหญ่ = อักขระใด ๆ ที่ไม่ใช่ช่องว่าง)
    #   ตัวเลือก (flag) 2 ตัว นำมารวมกันด้วยเครื่องหมาย | :
    #     - re.M (MULTILINE)   : ให้ ^ หมายถึง "ต้นบรรทัดทุกบรรทัด" ไม่ใช่เฉพาะต้นข้อความทั้งหมด
    #     - re.I (IGNORECASE)  : ค้นหาโดยไม่สนใจตัวพิมพ์เล็ก/ใหญ่ (HEMOGLOBIN ก็พบได้)
    pattern = re.compile(rf"^\s*({names})\s+(\d+(?:\.\d+)?)\s+(\S+)", re.M | re.I)
    # สร้าง "พจนานุกรม" (dict) จับคู่ ชื่อตัวพิมพ์เล็ก → ชื่อที่สะกดตามรายการ แล้วเก็บในตัวแปร spelling
    #   - dict { คีย์: ค่า } คือโครงสร้างข้อมูลแบบ "ค้นหาด้วยคีย์" เหมือนสมุดค้นคำ
    #   - เขียนแบบ dict comprehension : {คีย์: ค่า for n in KNOWN_TESTS} คือสร้างคู่ให้ทุกชื่อในรายการ
    #   - n.lower() : แปลงชื่อเป็นตัวพิมพ์เล็กทั้งหมด ใช้เป็นคีย์ เช่น "hemoglobin" → "Hemoglobin"
    #   ประโยชน์: เพราะค้นแบบไม่สนตัวพิมพ์ (re.I) หากเอกสารเขียน HEMOGLOBIN ก็จะถูกแปลงกลับเป็นตัวสะกดมาตรฐาน
    spelling = {n.lower(): n for n in KNOWN_TESTS}                   # ใช้ตัวสะกดของชื่อตามรายการที่กำหนด
    # ค้นหาทุกบรรทัดที่ตรงรูปแบบ แล้วแปลงผลเป็นรายการของแถวข้อมูล เก็บไว้ในตัวแปร rows
    #   - pattern.findall(text) : ค้นหาทุกตำแหน่งในข้อความ text ที่ตรงกับรูปแบบ
    #                             ได้ผลเป็นรายการของชุด 3 ค่า (กลุ่มที่ 1, 2, 3) เช่น ("Creatinine", "1.8", "mg/dL")
    #   - for t, v, u in ... : วนหยิบทีละชุด แล้วแยก 3 ค่าเก็บในตัวแปร t (ชื่อ), v (ค่า), u (หน่วย)
    #   - spelling[t.lower()] : แปลงชื่อเป็นตัวพิมพ์เล็กแล้วค้นใน dict เพื่อได้ตัวสะกดมาตรฐาน
    #                           (dict[คีย์] คือการดึงค่าที่จับคู่กับคีย์นั้น)
    #   - float(v)            : แปลงข้อความตัวเลข เช่น "1.8" ให้เป็นตัวเลขทศนิยมที่นำไปคำนวณ/เปรียบเทียบได้
    #   - u                   : หน่วย เก็บเป็นข้อความตามเดิม
    #   - ( ... , ... , ... ) : รวม 3 ค่าเป็น 1 แถว (tuple) และ [ ... ] รวมทุกแถวเป็นรายการ (list)
    rows = [(spelling[t.lower()], float(v), u) for t, v, u in pattern.findall(text)]
    # แปลงรายการแถวเป็นตาราง แล้วส่งกลับเป็นผลลัพธ์ของฟังก์ชัน (return)
    #   - pd.DataFrame(...) : ฟังก์ชันของไลบรารี pandas สร้างตารางข้อมูล (DataFrame = ตารางแบบมีแถวและคอลัมน์)
    #     - rows    : ข้อมูลของแต่ละแถว
    #     - columns=["test", "result", "unit"] : ตั้งชื่อคอลัมน์ทั้ง 3 ตามลำดับของค่าในแต่ละแถว
    return pd.DataFrame(rows, columns=["test", "result", "unit"])

# ----------------------------------------------------------------------------
# ส่วนที่ 3: เรียกใช้ฟังก์ชันกับข้อความจาก OCR และแสดงตารางผลลัพธ์
# ----------------------------------------------------------------------------
# เรียกฟังก์ชัน extract_labs กับข้อความ ocr_text (ผล OCR จากขั้นตอนที่ 2) แล้วเก็บตารางผลไว้ในตัวแปร labs
labs = extract_labs(ocr_text)
# เขียนชื่อตัวแปรไว้บรรทัดสุดท้ายของเซลล์ : Jupyter/Colab จะแสดงค่าของตัวแปรนั้นให้อัตโนมัติ
#   (สำหรับ DataFrame จะแสดงเป็นตารางสวยงาม โดยไม่ต้องใช้ print)
labs

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 2)
#   เปลี่ยน "ข้อความ" ที่ได้จาก OCR ให้เป็น "ตาราง" ผลการตรวจทางห้องปฏิบัติการ
#   (การสกัดสารสนเทศ — information extraction) โดยใช้รูปแบบการค้นหาข้อความ
#   (regular expression หรือ regex — นิพจน์ปรกติ) ค้นหาบรรทัดที่มีลักษณะ "ชื่อการตรวจ  ผล  หน่วย"
#   ข้อมูลนำเข้า : ocr_text (ข้อความจาก OCR) และรายชื่อการตรวจที่รู้จัก KNOWN_TESTS
#   ผลลัพธ์      : ตาราง labs 4 แถว มีคอลัมน์ test (ชื่อการตรวจ), result (ผล), unit (หน่วย)
#   คำตอบของแบบฝึกหัด: ชื่อที่ขาดไปคือ "LDL cholesterol" ซึ่งต้องเพิ่มลงใน KNOWN_TESTS
#   เพราะรูปแบบการค้นหานี้ค้นได้เฉพาะชื่อที่อยู่ในรายการเท่านั้น
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: รายชื่อการตรวจที่ต้องการค้นหา (ครบ 4 รายการตามรายงาน)
# ----------------------------------------------------------------------------
# สร้างตัวแปร KNOWN_TESTS เก็บ "รายการ" (list) ชื่อการตรวจที่ระบบรู้จัก
#   - list คือกลุ่มของค่าหลายค่าเรียงกัน เขียนไว้ใน [ ] และคั่นแต่ละค่าด้วยเครื่องหมาย ,
#   - แต่ละชื่อเป็นข้อความ จึงต้องอยู่ในเครื่องหมายคำพูด " "
#   - "LDL cholesterol" คือชื่อที่เพิ่มเข้ามา (เดิมขาดไป ทำให้ผล LDL 165 mg/dL ไม่ถูกสกัดออกมา)
KNOWN_TESTS = ["Hemoglobin", "Glucose (fasting)", "Creatinine", "LDL cholesterol"]

# ----------------------------------------------------------------------------
# ส่วนที่ 2: สร้างฟังก์ชัน extract_labs สำหรับสกัดผลการตรวจจากข้อความ
#   - def คือคำสั่งสร้างฟังก์ชัน (ชุดคำสั่งที่ตั้งชื่อไว้และเรียกใช้ซ้ำได้)
#   - text : พารามิเตอร์ คือข้อความ (เช่น ผลจาก OCR) ที่ต้องการค้นหาผลการตรวจ
#   - ผลลัพธ์ (return) : ตาราง (DataFrame) ที่มี 3 คอลัมน์ คือ test (ชื่อการตรวจ),
#     result (ผลการตรวจเป็นตัวเลข) และ unit (หน่วย)
# ----------------------------------------------------------------------------
def extract_labs(text):
    """ค้นหา 'ชื่อการตรวจ  ผล  หน่วย' ในแต่ละบรรทัดของข้อความ สำหรับทุกชื่อใน KNOWN_TESTS"""
    # (ข้อความในเครื่องหมาย """ ด้านบนเรียกว่า docstring คือคำอธิบายสั้น ๆ ว่าฟังก์ชันนี้ทำอะไร)
    # นำชื่อการตรวจทุกชื่อใน KNOWN_TESTS มาต่อกันเป็นข้อความเดียว คั่นด้วย | แล้วเก็บในตัวแปร names
    #   เช่น ได้ผลประมาณ "Hemoglobin|Glucose\ \(fasting\)|Creatinine"
    #   - for n in KNOWN_TESTS : วนหยิบชื่อการตรวจทีละชื่อมาเก็บในตัวแปร n
    #   - re.escape(n)         : ฟังก์ชันของไลบรารี re ที่ "ใส่เครื่องหมาย \ หน้าสัญลักษณ์พิเศษ"
    #                            เช่น วงเล็บใน "Glucose (fasting)" เพื่อให้ regex ถือเป็นตัวอักษรธรรมดา
    #                            (ปกติ ( ) ใน regex มีความหมายพิเศษว่า "กลุ่มที่ต้องการเก็บ")
    #   - "|".join(...)        : นำทุกชื่อมาต่อกัน โดยใช้ | คั่นกลาง; ใน regex เครื่องหมาย | หมายถึง "หรือ"
    names = "|".join(re.escape(n) for n in KNOWN_TESTS)            # เชื่อมชื่อการตรวจด้วย | (หมายถึง "หรือ")
    # สร้าง "รูปแบบการค้นหา" (regular expression) แล้วเก็บไว้ในตัวแปร pattern
    #   - re.compile(รูปแบบ, ตัวเลือก) : ฟังก์ชันของไลบรารี re ที่แปลงข้อความรูปแบบให้พร้อมใช้ค้นหาซ้ำได้
    #   - rf"..." : r = raw string (ให้ \ เป็นตัวอักษรธรรมดา ไม่ใช่สัญลักษณ์พิเศษของ Python)
    #               f = f-string (แทรกค่าตัวแปร names ลงในตำแหน่ง {names})
    #   ความหมายของรูปแบบทีละส่วน (อ่านจากซ้ายไปขวา):
    #     ^                : ต้องเริ่มที่ต้นบรรทัด
    #     \s*              : ตามด้วยช่องว่าง 0 ช่องขึ้นไป (\s = ช่องว่าง/แท็บ, * = มีกี่ตัวก็ได้รวมถึงไม่มี)
    #     ({names})        : กลุ่มที่ 1 = ชื่อการตรวจชื่อใดชื่อหนึ่งในรายการ (วงเล็บ ( ) = ส่วนที่ต้องการเก็บไว้)
    #     \s+              : ตามด้วยช่องว่างอย่างน้อย 1 ช่อง (+ = 1 ตัวขึ้นไป)
    #     (\d+(?:\.\d+)?)  : กลุ่มที่ 2 = ผลการตรวจที่เป็นตัวเลข
    #                          \d+       = ตัวเลข 0-9 อย่างน้อย 1 หลัก เช่น 142
    #                          (?:\.\d+) = กลุ่ม "จุดทศนิยมตามด้วยตัวเลข" เช่น .8
    #                                      (?: ) คือการจัดกลุ่มโดยไม่เก็บแยก; \. คือจุดจริง ๆ
    #                          ?         = กลุ่มทศนิยมนี้จะมีหรือไม่มีก็ได้ (จึงรับทั้ง 142 และ 1.8)
    #     \s+              : ตามด้วยช่องว่างอย่างน้อย 1 ช่อง
    #     (\S+)            : กลุ่มที่ 3 = หน่วย คืออักขระที่ไม่ใช่ช่องว่างติดกันอย่างน้อย 1 ตัว เช่น mg/dL
    #                        (\S ตัวใหญ่ = อักขระใด ๆ ที่ไม่ใช่ช่องว่าง)
    #   ตัวเลือก (flag) 2 ตัว นำมารวมกันด้วยเครื่องหมาย | :
    #     - re.M (MULTILINE)   : ให้ ^ หมายถึง "ต้นบรรทัดทุกบรรทัด" ไม่ใช่เฉพาะต้นข้อความทั้งหมด
    #     - re.I (IGNORECASE)  : ค้นหาโดยไม่สนใจตัวพิมพ์เล็ก/ใหญ่ (HEMOGLOBIN ก็พบได้)
    pattern = re.compile(rf"^\s*({names})\s+(\d+(?:\.\d+)?)\s+(\S+)", re.M | re.I)
    # สร้าง "พจนานุกรม" (dict) จับคู่ ชื่อตัวพิมพ์เล็ก → ชื่อที่สะกดตามรายการ แล้วเก็บในตัวแปร spelling
    #   - dict { คีย์: ค่า } คือโครงสร้างข้อมูลแบบ "ค้นหาด้วยคีย์" เหมือนสมุดค้นคำ
    #   - เขียนแบบ dict comprehension : {คีย์: ค่า for n in KNOWN_TESTS} คือสร้างคู่ให้ทุกชื่อในรายการ
    #   - n.lower() : แปลงชื่อเป็นตัวพิมพ์เล็กทั้งหมด ใช้เป็นคีย์ เช่น "hemoglobin" → "Hemoglobin"
    #   ประโยชน์: เพราะค้นแบบไม่สนตัวพิมพ์ (re.I) หากเอกสารเขียน HEMOGLOBIN ก็จะถูกแปลงกลับเป็นตัวสะกดมาตรฐาน
    spelling = {n.lower(): n for n in KNOWN_TESTS}                   # ใช้ตัวสะกดของชื่อตามรายการที่กำหนด
    # ค้นหาทุกบรรทัดที่ตรงรูปแบบ แล้วแปลงผลเป็นรายการของแถวข้อมูล เก็บไว้ในตัวแปร rows
    #   - pattern.findall(text) : ค้นหาทุกตำแหน่งในข้อความ text ที่ตรงกับรูปแบบ
    #                             ได้ผลเป็นรายการของชุด 3 ค่า (กลุ่มที่ 1, 2, 3) เช่น ("Creatinine", "1.8", "mg/dL")
    #   - for t, v, u in ... : วนหยิบทีละชุด แล้วแยก 3 ค่าเก็บในตัวแปร t (ชื่อ), v (ค่า), u (หน่วย)
    #   - spelling[t.lower()] : แปลงชื่อเป็นตัวพิมพ์เล็กแล้วค้นใน dict เพื่อได้ตัวสะกดมาตรฐาน
    #                           (dict[คีย์] คือการดึงค่าที่จับคู่กับคีย์นั้น)
    #   - float(v)            : แปลงข้อความตัวเลข เช่น "1.8" ให้เป็นตัวเลขทศนิยมที่นำไปคำนวณ/เปรียบเทียบได้
    #   - u                   : หน่วย เก็บเป็นข้อความตามเดิม
    #   - ( ... , ... , ... ) : รวม 3 ค่าเป็น 1 แถว (tuple) และ [ ... ] รวมทุกแถวเป็นรายการ (list)
    rows = [(spelling[t.lower()], float(v), u) for t, v, u in pattern.findall(text)]
    # แปลงรายการแถวเป็นตาราง แล้วส่งกลับเป็นผลลัพธ์ของฟังก์ชัน (return)
    #   - pd.DataFrame(...) : ฟังก์ชันของไลบรารี pandas สร้างตารางข้อมูล (DataFrame = ตารางแบบมีแถวและคอลัมน์)
    #     - rows    : ข้อมูลของแต่ละแถว
    #     - columns=["test", "result", "unit"] : ตั้งชื่อคอลัมน์ทั้ง 3 ตามลำดับของค่าในแต่ละแถว
    return pd.DataFrame(rows, columns=["test", "result", "unit"])

# ----------------------------------------------------------------------------
# ส่วนที่ 3: เรียกใช้ฟังก์ชันกับข้อความจาก OCR และแสดงตารางผลลัพธ์
# ----------------------------------------------------------------------------
# เรียกฟังก์ชัน extract_labs กับข้อความ ocr_text (ผล OCR จากขั้นตอนที่ 2) แล้วเก็บตารางผลไว้ในตัวแปร labs
labs = extract_labs(ocr_text)
# เขียนชื่อตัวแปรไว้บรรทัดสุดท้ายของเซลล์ : Jupyter/Colab จะแสดงค่าของตัวแปรนั้นให้อัตโนมัติ
#   (สำหรับ DataFrame จะแสดงเป็นตารางสวยงาม โดยไม่ต้องใช้ print)
labs

💡 **คำอธิบายผลลัพธ์**

ข้อความที่ไม่มีโครงสร้างถูกแปลงเป็นตาราง โดยแต่ละแถวแทนการตรวจหนึ่งรายการ ข้อมูลในรูปแบบนี้สามารถนำไปเรียงลำดับ นำเสนอเป็นกราฟ หรือส่งเข้าสู่เวชระเบียนอิเล็กทรอนิกส์ (EHR) ได้ ตัวเลือก `re.I` กำหนดให้การค้นหาไม่คำนึงถึงตัวพิมพ์เล็กและตัวพิมพ์ใหญ่ ดังนั้น `HEMOGLOBIN` จะถูกค้นพบด้วย


## ขั้นตอนที่ 4 — การระบุค่าผิดปกติ

ระบบเปรียบเทียบผลการตรวจแต่ละรายการกับช่วงค่าอ้างอิง (reference range) ในคอลัมน์ REFERENCE ของรายงาน โดย `LOW` หมายถึงต่ำกว่าช่วงค่าอ้างอิง และ `HIGH` หมายถึงสูงกว่าช่วงค่าอ้างอิง


In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: การเพิ่มช่วงค่าอ้างอิงที่ขาดไป
# แต่ละรายการมีรูปแบบ  "ชื่อการตรวจ": (ค่าปกติต่ำสุด, ค่าปกติสูงสุด)
# รายการยังไม่มี Creatinine ผลลัพธ์จึงแสดงว่า "no range given" (ไม่มีช่วงค่าอ้างอิง)
# ➜ ให้อ่านช่วงค่าอ้างอิงของ Creatinine จากภาพรายงาน (คอลัมน์ REFERENCE) และเพิ่มไว้ด้านล่าง
#   ในรูปแบบเดียวกับบรรทัดอื่น จากนั้นรันเซลล์ ผลลัพธ์ที่ถูกต้องคือ Creatinine 1.8 ถูกระบุเป็น HIGH
# (สำหรับ LDL รายงานระบุว่า "<100" หมายความว่าค่า 0 ถึง 99 อยู่ในเกณฑ์ปกติ)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ระบุค่าผิดปกติ (abnormal flag) ของผลการตรวจแต่ละรายการ โดยเทียบกับช่วงค่าอ้างอิง
#   (reference range) แล้วติดป้าย LOW (ต่ำกว่าเกณฑ์) / HIGH (สูงกว่าเกณฑ์) / normal (ปกติ)
#   เป็นหลักการเดียวกับการแสดงผลแล็บเป็นสีแดงในระบบสารสนเทศห้องปฏิบัติการ (LIS) หรือ EHR
#   ข้อมูลนำเข้า : ตาราง labs จากขั้นตอนที่ 3 และช่วงค่าอ้างอิงใน REF_RANGES
#   ผลลัพธ์      : ตาราง labs ที่มีคอลัมน์ใหม่ชื่อ flag
#   สิ่งที่ผู้เรียนต้องทำ: เพิ่มช่วงค่าอ้างอิงของ Creatinine ใน REF_RANGES ตามคอลัมน์ REFERENCE ของภาพรายงาน
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: ช่วงค่าอ้างอิงของการตรวจแต่ละรายการ
# ----------------------------------------------------------------------------
# สร้างตัวแปร REF_RANGES เป็น "พจนานุกรม" (dict) จับคู่ ชื่อการตรวจ → ช่วงค่าอ้างอิง
#   - dict เขียนใน { } แต่ละคู่อยู่ในรูป คีย์: ค่า และคั่นแต่ละคู่ด้วยเครื่องหมาย ,
#   - คีย์ (ซ้ายของ :) คือชื่อการตรวจ ต้องสะกดตรงกับคอลัมน์ test ในตาราง labs
#   - ค่า (ขวาของ :) คือ tuple (ค่าปกติต่ำสุด, ค่าปกติสูงสุด) เขียนในวงเล็บ ( )
#   - ข้อความหลัง # ท้ายบรรทัด คือหน่วยของค่า (เป็นคำอธิบาย ไม่มีผลต่อการทำงาน)
#   - คำสั่งนี้ยาวหลายบรรทัด Python จะอ่านต่อเนื่องจนพบวงเล็บปิด }
REF_RANGES = {
    # ฮีโมโกลบิน: ปกติ 13.5 ถึง 17.5 g/dL (ค่าอ้างอิงสำหรับเพศชายวัยผู้ใหญ่)
    "Hemoglobin":        (13.5, 17.5),   # g/dL (เพศชายวัยผู้ใหญ่)
    # น้ำตาลในเลือดขณะอดอาหาร: ปกติ 70 ถึง 99 mg/dL
    "Glucose (fasting)": (70, 99),       # mg/dL
    # LDL cholesterol: รายงานระบุ "<100" จึงใช้ 0 ถึง 99 mg/dL
    "LDL cholesterol":   (0, 99),        # mg/dL
    # ตำแหน่งที่ผู้เรียนต้องเพิ่มบรรทัดของ Creatinine ในรูปแบบเดียวกับบรรทัดด้านบน
    #   (อย่าลืมเครื่องหมายคำพูดรอบชื่อ, เครื่องหมาย : , วงเล็บ ( ) และจุลภาค , ท้ายบรรทัด)
    # TODO เพิ่ม Creatinine ในบรรทัดนี้
}

# ----------------------------------------------------------------------------
# ส่วนที่ 2: สร้างฟังก์ชัน flag สำหรับตัดสินว่าผลการตรวจ 1 แถว ต่ำ / สูง / ปกติ
#   - def คือคำสั่งสร้างฟังก์ชัน
#   - row : พารามิเตอร์ คือข้อมูล 1 แถวของตาราง labs (มีค่าในคอลัมน์ test, result, unit)
#   - ผลลัพธ์ (return) : ข้อความ "LOW", "HIGH", "normal" หรือ "no range given"
#   - เมื่อฟังก์ชันพบคำสั่ง return จะส่งค่ากลับและหยุดทำงานทันที (บรรทัดที่เหลือจะไม่ถูกทำ)
# ----------------------------------------------------------------------------
def flag(row):
    # ตรวจว่าชื่อการตรวจของแถวนี้ "ไม่มี" อยู่ใน REF_RANGES ใช่หรือไม่
    #   - row["test"]    : ดึงค่าในคอลัมน์ test ของแถวนี้ (ชื่อการตรวจ) ด้วยการเขียนชื่อคอลัมน์ใน [ ]
    #   - not in         : ตรวจว่า "ไม่อยู่ใน" ; กับ dict จะตรวจจากคีย์ (ชื่อการตรวจ)
    if row["test"] not in REF_RANGES:
        # ถ้าไม่มีช่วงค่าอ้างอิง ให้ส่งกลับข้อความ "no range given" (ไม่มีช่วงค่าอ้างอิง) แล้วจบฟังก์ชัน
        return "no range given"
    # ดึงช่วงค่าอ้างอิงของการตรวจนี้จาก REF_RANGES แล้วแยกเก็บในตัวแปร 2 ตัว
    #   - REF_RANGES[row["test"]] : ค้นใน dict ด้วยชื่อการตรวจ ได้ผลเป็นคู่ค่า (ต่ำสุด, สูงสุด)
    #   - low, high = ...         : แยกคู่ค่าออก ค่าแรกเก็บใน low ค่าที่สองเก็บใน high
    low, high = REF_RANGES[row["test"]]
    # ถ้าผลการตรวจ (คอลัมน์ result) น้อยกว่าค่าต่ำสุดของช่วงอ้างอิง ( < คือ "น้อยกว่า")
    if row["result"] < low:
        # ส่งกลับ "LOW" (ต่ำกว่าเกณฑ์ปกติ)
        return "LOW"
    # ถ้าผลการตรวจมากกว่าค่าสูงสุดของช่วงอ้างอิง ( > คือ "มากกว่า")
    if row["result"] > high:
        # ส่งกลับ "HIGH" (สูงกว่าเกณฑ์ปกติ)
        return "HIGH"
    # ถ้าไม่เข้าเงื่อนไขใดด้านบนเลย แสดงว่าอยู่ในช่วงอ้างอิง (รวมค่าที่เท่ากับขอบพอดี) จึงส่งกลับ "normal" (ปกติ)
    return "normal"

# ----------------------------------------------------------------------------
# ส่วนที่ 3: ใช้ฟังก์ชัน flag กับทุกแถวของตาราง labs แล้วเก็บผลเป็นคอลัมน์ใหม่
# ----------------------------------------------------------------------------
# สร้างคอลัมน์ใหม่ชื่อ "flag" ในตาราง labs (ตารางจากขั้นตอนที่ 3)
#   - labs.apply(flag, axis=1) : คำสั่งของ pandas ที่เรียกฟังก์ชันซ้ำกับข้อมูลในตาราง
#     - flag   : ฟังก์ชันที่จะเรียกใช้ (ส่งชื่อฟังก์ชันไปเฉย ๆ โดยไม่ใส่วงเล็บ)
#     - axis=1 : ให้ส่งข้อมูล "ทีละแถว" เข้าไปในฟังก์ชัน (axis=0 จะเป็นทีละคอลัมน์)
#   - labs["flag"] = ... : นำผลลัพธ์ของทุกแถวไปเก็บเป็นคอลัมน์ชื่อ flag (ถ้ายังไม่มีจะสร้างใหม่)
labs["flag"] = labs.apply(flag, axis=1)
# ชื่อตัวแปรบรรทัดสุดท้ายของเซลล์ : Jupyter/Colab แสดงตาราง labs (ซึ่งมีคอลัมน์ flag เพิ่มแล้ว) ให้อัตโนมัติ
labs

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 3)
#   ระบุค่าผิดปกติ (abnormal flag) ของผลการตรวจแต่ละรายการ โดยเทียบกับช่วงค่าอ้างอิง
#   (reference range) แล้วติดป้าย LOW (ต่ำกว่าเกณฑ์) / HIGH (สูงกว่าเกณฑ์) / normal (ปกติ)
#   ข้อมูลนำเข้า : ตาราง labs จากขั้นตอนที่ 3 และช่วงค่าอ้างอิงใน REF_RANGES
#   ผลลัพธ์      : ตาราง labs ที่มีคอลัมน์ flag เช่น Hemoglobin 11.2 → LOW, Creatinine 1.8 → HIGH
#   คำตอบของแบบฝึกหัด: เพิ่ม "Creatinine": (0.7, 1.3) ตามคอลัมน์ REFERENCE ของรายงาน (0.7-1.3 mg/dL)
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: ช่วงค่าอ้างอิงของการตรวจแต่ละรายการ (ครบ 4 รายการ)
# ----------------------------------------------------------------------------
# สร้างตัวแปร REF_RANGES เป็น "พจนานุกรม" (dict) จับคู่ ชื่อการตรวจ → ช่วงค่าอ้างอิง
#   - dict เขียนใน { } แต่ละคู่อยู่ในรูป คีย์: ค่า และคั่นแต่ละคู่ด้วยเครื่องหมาย ,
#   - คีย์ (ซ้ายของ :) คือชื่อการตรวจ ต้องสะกดตรงกับคอลัมน์ test ในตาราง labs
#   - ค่า (ขวาของ :) คือ tuple (ค่าปกติต่ำสุด, ค่าปกติสูงสุด) เขียนในวงเล็บ ( )
#   - ข้อความหลัง # ท้ายบรรทัด คือหน่วยของค่า (เป็นคำอธิบาย ไม่มีผลต่อการทำงาน)
REF_RANGES = {
    # ฮีโมโกลบิน: ปกติ 13.5 ถึง 17.5 g/dL (ค่าอ้างอิงสำหรับเพศชายวัยผู้ใหญ่)
    "Hemoglobin":        (13.5, 17.5),   # g/dL (เพศชายวัยผู้ใหญ่)
    # น้ำตาลในเลือดขณะอดอาหาร: ปกติ 70 ถึง 99 mg/dL
    "Glucose (fasting)": (70, 99),       # mg/dL
    # LDL cholesterol: รายงานระบุ "<100" จึงใช้ 0 ถึง 99 mg/dL
    "LDL cholesterol":   (0, 99),        # mg/dL
    # ครีอะตินีน (creatinine): ปกติ 0.7 ถึง 1.3 mg/dL — บรรทัดที่เพิ่มเข้ามาตามเฉลย
    "Creatinine":        (0.7, 1.3),     # mg/dL
}

# ----------------------------------------------------------------------------
# ส่วนที่ 2: สร้างฟังก์ชัน flag สำหรับตัดสินว่าผลการตรวจ 1 แถว ต่ำ / สูง / ปกติ
#   - def คือคำสั่งสร้างฟังก์ชัน
#   - row : พารามิเตอร์ คือข้อมูล 1 แถวของตาราง labs (มีค่าในคอลัมน์ test, result, unit)
#   - ผลลัพธ์ (return) : ข้อความ "LOW", "HIGH", "normal" หรือ "no range given"
#   - เมื่อฟังก์ชันพบคำสั่ง return จะส่งค่ากลับและหยุดทำงานทันที (บรรทัดที่เหลือจะไม่ถูกทำ)
# ----------------------------------------------------------------------------
def flag(row):
    # ตรวจว่าชื่อการตรวจของแถวนี้ "ไม่มี" อยู่ใน REF_RANGES ใช่หรือไม่
    #   - row["test"]    : ดึงค่าในคอลัมน์ test ของแถวนี้ (ชื่อการตรวจ) ด้วยการเขียนชื่อคอลัมน์ใน [ ]
    #   - not in         : ตรวจว่า "ไม่อยู่ใน" ; กับ dict จะตรวจจากคีย์ (ชื่อการตรวจ)
    if row["test"] not in REF_RANGES:
        # ถ้าไม่มีช่วงค่าอ้างอิง ให้ส่งกลับข้อความ "no range given" (ไม่มีช่วงค่าอ้างอิง) แล้วจบฟังก์ชัน
        return "no range given"
    # ดึงช่วงค่าอ้างอิงของการตรวจนี้จาก REF_RANGES แล้วแยกเก็บในตัวแปร 2 ตัว
    #   - REF_RANGES[row["test"]] : ค้นใน dict ด้วยชื่อการตรวจ ได้ผลเป็นคู่ค่า (ต่ำสุด, สูงสุด)
    #   - low, high = ...         : แยกคู่ค่าออก ค่าแรกเก็บใน low ค่าที่สองเก็บใน high
    low, high = REF_RANGES[row["test"]]
    # ถ้าผลการตรวจ (คอลัมน์ result) น้อยกว่าค่าต่ำสุดของช่วงอ้างอิง ( < คือ "น้อยกว่า")
    if row["result"] < low:
        # ส่งกลับ "LOW" (ต่ำกว่าเกณฑ์ปกติ)
        return "LOW"
    # ถ้าผลการตรวจมากกว่าค่าสูงสุดของช่วงอ้างอิง ( > คือ "มากกว่า")
    if row["result"] > high:
        # ส่งกลับ "HIGH" (สูงกว่าเกณฑ์ปกติ)
        return "HIGH"
    # ถ้าไม่เข้าเงื่อนไขใดด้านบนเลย แสดงว่าอยู่ในช่วงอ้างอิง (รวมค่าที่เท่ากับขอบพอดี) จึงส่งกลับ "normal" (ปกติ)
    return "normal"

# ----------------------------------------------------------------------------
# ส่วนที่ 3: ใช้ฟังก์ชัน flag กับทุกแถวของตาราง labs แล้วเก็บผลเป็นคอลัมน์ใหม่
# ----------------------------------------------------------------------------
# สร้างคอลัมน์ใหม่ชื่อ "flag" ในตาราง labs (ตารางจากขั้นตอนที่ 3)
#   - labs.apply(flag, axis=1) : คำสั่งของ pandas ที่เรียกฟังก์ชันซ้ำกับข้อมูลในตาราง
#     - flag   : ฟังก์ชันที่จะเรียกใช้ (ส่งชื่อฟังก์ชันไปเฉย ๆ โดยไม่ใส่วงเล็บ)
#     - axis=1 : ให้ส่งข้อมูล "ทีละแถว" เข้าไปในฟังก์ชัน (axis=0 จะเป็นทีละคอลัมน์)
#   - labs["flag"] = ... : นำผลลัพธ์ของทุกแถวไปเก็บเป็นคอลัมน์ชื่อ flag (ถ้ายังไม่มีจะสร้างใหม่)
labs["flag"] = labs.apply(flag, axis=1)
# ชื่อตัวแปรบรรทัดสุดท้ายของเซลล์ : Jupyter/Colab แสดงตาราง labs (ซึ่งมีคอลัมน์ flag เพิ่มแล้ว) ให้อัตโนมัติ
labs

💡 **คำอธิบายผลลัพธ์**

กฎอย่างง่ายแปลงค่าตัวเลขเป็นการระบุค่าผิดปกติทางคลินิก ซึ่งเป็นหลักการเดียวกับการแสดงผลการตรวจเป็นสีแดงในระบบสารสนเทศห้องปฏิบัติการ (LIS) หรือ EHR อย่างไรก็ตาม ช่วงค่าอ้างอิงขึ้นกับอายุ เพศ และวิธีการตรวจของแต่ละห้องปฏิบัติการ ระบบที่ใช้งานจริงจึงควรอ่านช่วงค่าอ้างอิงที่ห้องปฏิบัติการ *นั้น* ระบุไว้ แทนการกำหนดค่าคงที่ในโปรแกรม

## ขั้นตอนที่ 5 — การสกัดรายการยา

ความหมายของรูปแบบการค้นหายาสำเร็จรูปมีดังนี้

| ส่วนของรูปแบบ | ความหมาย |
|---|---|
| `([A-Z][a-z]+)` | คำที่ขึ้นต้นด้วยอักษรตัวพิมพ์ใหญ่ เช่น `Metformin` (**ชื่อยา**) … |
| `\s+(\d+(?:\.\d+)?)` | … ตามด้วยช่องว่างและตัวเลข เช่น `500` หรือ `12.5` (**ขนาดยา**) … |
| `\s*mg(?![A-Za-z/])` | … ตามด้วยตัวอักษร `mg` (อาจมีช่องว่างคั่น) แต่ **ไม่รวม** `mg/dL` ซึ่งเป็นหน่วยของผลการตรวจ มิใช่ขนาดยา |


In [ ]:
# รูปแบบการค้นหายา: ชื่อยา + ขนาดยาในหน่วย mg
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   สกัดรายการยา (ชื่อยาและขนาดยาในหน่วย mg) ออกจากข้อความที่ได้จาก OCR
#   โดยใช้รูปแบบการค้นหาข้อความ (regular expression) ที่เตรียมไว้
#   ข้อมูลนำเข้า : ocr_text (ข้อความจาก OCR ของรายงาน)
#   ผลลัพธ์      : ตาราง meds มี 2 คอลัมน์ คือ name (ชื่อยา) และ dose_mg (ขนาดยาเป็นมิลลิกรัม)
#                  สำหรับรายงานนี้ควรพบ Metformin 500 และ Atorvastatin 20
#   ข้อสังเกต: รูปแบบถูกออกแบบให้ "ไม่" นับ mg/dL (หน่วยของผลแล็บ) เป็นขนาดยา
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: สร้างรูปแบบการค้นหายา
# ----------------------------------------------------------------------------
# สร้างรูปแบบการค้นหา (regular expression) แล้วเก็บไว้ในตัวแปร MED_RE
#   - re.compile(...) : ฟังก์ชันของไลบรารี re แปลงข้อความรูปแบบให้พร้อมใช้ค้นหาซ้ำได้
#   - r"..."          : raw string ทำให้เครื่องหมาย \ ถูกส่งให้ regex ตามที่เขียน
#   ความหมายของรูปแบบทีละส่วน (อ่านจากซ้ายไปขวา):
#     ([A-Z][a-z]+)    : กลุ่มที่ 1 = ชื่อยา คือคำที่ขึ้นต้นด้วยอักษรอังกฤษตัวพิมพ์ใหญ่ 1 ตัว ([A-Z])
#                        ตามด้วยตัวพิมพ์เล็กอย่างน้อย 1 ตัว ([a-z]+) เช่น Metformin
#                        ([ ] = เลือกอักขระ 1 ตัวจากช่วงที่กำหนด, + = 1 ตัวขึ้นไป)
#     \s+              : ตามด้วยช่องว่างอย่างน้อย 1 ช่อง
#     (\d+(?:\.\d+)?)  : กลุ่มที่ 2 = ขนาดยา คือตัวเลขอย่างน้อย 1 หลัก (\d+)
#                        อาจมีทศนิยมหรือไม่ก็ได้ ((?:\.\d+)? ; \. = จุดทศนิยม, ? = มีหรือไม่มีก็ได้)
#                        เช่น 500 หรือ 12.5
#     \s*              : ตามด้วยช่องว่าง 0 ช่องขึ้นไป (จึงรับทั้ง "500 mg" และ "500mg")
#     mg               : ตามด้วยตัวอักษร mg ตรงตัว
#     (?![A-Za-z/])    : "มองไปข้างหน้าแบบปฏิเสธ" (negative lookahead) = หลัง mg ต้อง "ไม่" เป็น
#                        ตัวอักษรอังกฤษหรือเครื่องหมาย / เพื่อตัด mg/dL หรือ mgX ออก
#                        (ส่วนนี้แค่ตรวจ ไม่ได้นับรวมอักขระถัดไปเข้าไปในผลลัพธ์)
MED_RE = re.compile(r"([A-Z][a-z]+)\s+(\d+(?:\.\d+)?)\s*mg(?![A-Za-z/])")

# ----------------------------------------------------------------------------
# ส่วนที่ 2: สร้างฟังก์ชัน extract_meds สำหรับสกัดรายการยาจากข้อความ
#   - text : พารามิเตอร์ คือข้อความที่ต้องการค้นหายา
#   - ผลลัพธ์ (return) : ตาราง (DataFrame) คอลัมน์ name และ dose_mg
# ----------------------------------------------------------------------------
def extract_meds(text):
    # ค้นหายาทั้งหมดแล้วสร้างเป็นตารางส่งกลับ (ทำงานจากในออกนอก)
    #   1) MED_RE.findall(text)  : ค้นหาทุกตำแหน่งที่ตรงกับรูปแบบ ได้รายการของคู่ (ชื่อยา, ขนาดยา) เป็นข้อความ
    #   2) for n, d in ...       : วนหยิบทีละคู่ แยกเก็บในตัวแปร n (ชื่อยา) และ d (ขนาดยา)
    #   3) (n, float(d))         : สร้างแถวใหม่ โดยแปลงขนาดยาจากข้อความเป็นตัวเลขทศนิยมด้วย float(...)
    #   4) [ ... ]               : รวมทุกแถวเป็นรายการ (list comprehension)
    #   5) pd.DataFrame(..., columns=["name", "dose_mg"]) : ฟังก์ชันของ pandas สร้างตาราง
    #      - อาร์กิวเมนต์แรก : รายการแถวข้อมูล
    #      - columns         : ชื่อคอลัมน์ name (ชื่อยา) และ dose_mg (ขนาดยาเป็น mg)
    return pd.DataFrame([(n, float(d)) for n, d in MED_RE.findall(text)], columns=["name", "dose_mg"])

# ----------------------------------------------------------------------------
# ส่วนที่ 3: เรียกใช้ฟังก์ชันกับข้อความจาก OCR และแสดงผล
# ----------------------------------------------------------------------------
# เรียกฟังก์ชัน extract_meds กับข้อความ ocr_text แล้วเก็บตารางรายการยาไว้ในตัวแปร meds
meds = extract_meds(ocr_text)
# ชื่อตัวแปรบรรทัดสุดท้ายของเซลล์ : Jupyter/Colab แสดงตาราง meds ให้อัตโนมัติ
meds

In [ ]:
# ⭐ Exercise 4 — แบบฝึกหัดที่ 4: การทดสอบรูปแบบการค้นหายากับข้อความที่ผู้เรียนกำหนด
# ➜ ให้แทนข้อความภายในเครื่องหมายคำพูดด้วยบรรทัดรายการยาในลักษณะที่พบในหน่วยงานของผู้เรียน
#   เช่น "Medications: Amlodipine 5 mg OD; aspirin 81mg OD; Insulin glargine 20 units QHS"
#   ทดลองใช้ชื่อยาตัวพิมพ์เล็ก ขนาดยาในหน่วย units / g / mcg ชื่อการค้า ชื่อย่อ (HCTZ) หรือภาษาไทย
#   จากนั้นตรวจสอบว่ารูปแบบนี้ค้นพบยารายการใด และไม่พบยารายการใด
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ทดสอบ "ข้อจำกัด" ของรูปแบบการค้นหายา (MED_RE) ด้วยข้อความรายการยาที่ผู้เรียนเขียนเอง
#   ตามลักษณะที่พบจริงในหน่วยงาน เพื่อดูว่ากฎที่เขียนขึ้นเองพลาดยาแบบใดบ้าง
#   ข้อมูลนำเข้า : ข้อความรายการยาในตัวแปร my_line (ผู้เรียนกำหนดเอง)
#   ผลลัพธ์      : ตารางยาที่รูปแบบค้นพบ (ยาที่ไม่ปรากฏในตาราง คือยาที่รูปแบบค้นไม่พบ)
#   ผลนี้ใช้ตอบคำถามที่ 2 เรื่องผลกระทบต่อการทบทวนความสอดคล้องของรายการยา (medication reconciliation)
# ============================================================================

# สร้างตัวแปร my_line เก็บข้อความรายการยา 1 บรรทัด
#   - ข้อความต้องอยู่ภายในเครื่องหมายคำพูด " " เสมอ
#   - ... ภายในเครื่องหมายคำพูด คือส่วนที่ผู้เรียนต้องแทนที่ด้วยรายการยาของตนเอง
#     (ตัวอย่างรูปแบบอยู่ในคำแนะนำด้านบน; แยกยาแต่ละตัวด้วย ; ได้)
#   - # TODO ท้ายบรรทัด คือป้ายเตือนว่าบรรทัดนี้ต้องแก้ไข
my_line = "Medications: ..."   # TODO กำหนดข้อความรายการยา
# เรียกฟังก์ชัน extract_meds (สร้างไว้ในเซลล์ก่อนหน้า) เพื่อค้นหายาในข้อความ my_line
#   เนื่องจากเป็นบรรทัดสุดท้ายของเซลล์ Jupyter/Colab จะแสดงตารางผลลัพธ์ให้อัตโนมัติ
extract_meds(my_line)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ตัวอย่างเฉลยแบบฝึกหัดที่ 4)
#   ทดสอบรูปแบบการค้นหายา (MED_RE) กับข้อความรายการยาตัวอย่างที่มีรูปแบบหลากหลาย
#   ข้อมูลนำเข้า : ข้อความรายการยา 4 รายการในตัวแปร my_line
#   ผลลัพธ์      : ตารางที่พบเพียง Amlodipine 5 mg เท่านั้น เพราะ
#     - aspirin  : ขึ้นต้นด้วยตัวพิมพ์เล็ก (รูปแบบกำหนดให้ขึ้นต้นด้วยตัวพิมพ์ใหญ่ [A-Z])
#                  และ "81mg" ไม่มีช่องว่างระหว่างชื่อยากับตัวเลข (รูปแบบต้องมี \s+)
#     - Insulin glargine 20 units : หน่วยเป็น units ไม่ใช่ mg และหลังคำ Insulin ไม่ใช่ตัวเลข
#     - Metformin 1 g : หน่วยเป็น g ไม่ใช่ mg
#   ข้อคิดทางคลินิก: ยาที่ตกหล่นจากการสกัดอาจทำให้การทบทวนรายการยาไม่ครบถ้วน
# ============================================================================

# สร้างตัวแปร my_line เก็บข้อความรายการยาตัวอย่าง 4 รายการ คั่นด้วย ;
#   (OD = วันละ 1 ครั้ง, QHS = ก่อนนอน, BID = วันละ 2 ครั้ง)
my_line = "Medications: Amlodipine 5 mg OD; aspirin 81mg OD; Insulin glargine 20 units QHS; Metformin 1 g BID"
# เรียกฟังก์ชัน extract_meds ค้นหายาในข้อความ my_line
#   เป็นบรรทัดสุดท้ายของเซลล์ Jupyter/Colab จึงแสดงตารางผลลัพธ์ให้อัตโนมัติ
extract_meds(my_line)   # พบเพียง Amlodipine: 'aspirin' เป็นตัวพิมพ์เล็ก insulin ใช้หน่วย units และ metformin ใช้หน่วย g

✍️ **คำถามที่ 2** จากผลลัพธ์ข้างต้น รูปแบบการค้นหาไม่พบยารายการใดในข้อความของผู้เรียน เพราะเหตุใด (โปรดอ้างอิงตารางคำอธิบายรูปแบบ) และหากนำรายการยาที่สกัดได้ในลักษณะนี้ไปใช้ในการทบทวนความสอดคล้องของรายการยา (medication reconciliation) อาจเกิดปัญหาทางคลินิกใด


✍️ **คำตอบ:**



## ขั้นตอนที่ 6 — การประเมินความถูกต้องของการสกัดสารสนเทศ และผลกระทบของข้อผิดพลาดจาก OCR

การประเมินความถูกต้องทำโดยเปรียบเทียบคู่ข้อมูล **(ชื่อการตรวจ, ผลการตรวจ)** ที่สกัดได้กับเฉลย ด้วยตัวชี้วัดต่อไปนี้

- **Precision** (ค่าพยากรณ์บวก) — สัดส่วนของค่าที่ระบบสกัดได้ซึ่งถูกต้อง (precision ต่ำ หมายถึงมีค่าที่ผิดถูกบันทึกลงในเวชระเบียน)
- **Recall** (ความไว หรือ sensitivity) — สัดส่วนของค่าที่มีอยู่จริงในรายงานซึ่งระบบค้นพบ (recall ต่ำ หมายถึงผลการตรวจตกหล่น)
- **F1** — ค่าเฉลี่ยฮาร์มอนิกของ precision และ recall (0 = ใช้งานไม่ได้, 1 = ถูกต้องสมบูรณ์) ตารางอันดับ 🏆 ใช้หลักการเดียวกันนี้

คู่ข้อมูลจะนับว่าถูกต้องเมื่อทั้งชื่อ **และ** ตัวเลขถูกต้องทุกหลักเท่านั้น เช่น ค่า `1.8` ที่อ่านเป็น `18` ถือว่าผิด ข้อความด้านล่างคือรายงานฉบับเดิมในรูปแบบ *ที่มีสัญญาณรบกวน* ซึ่งจำลองผล OCR ของเอกสารแฟกซ์คุณภาพต่ำ


In [ ]:
# เฉลยของรายงานในขั้นตอนที่ 1
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   วัดความถูกต้องของการสกัดผลการตรวจ และแสดงให้เห็นว่าข้อผิดพลาดจาก OCR ส่งผลอย่างไร
#   เซลล์นี้เตรียมเครื่องมือ 4 อย่าง ได้แก่
#     1) ANSWER_LABS      : เฉลยคู่ (ชื่อการตรวจ, ผลการตรวจ) ที่ถูกต้องของรายงาน
#     2) pair_f1(...)     : ฟังก์ชันคำนวณตัวชี้วัด precision (ค่าพยากรณ์บวก), recall (ความไว — sensitivity)
#                           และ F1 (ค่าเฉลี่ยฮาร์มอนิกของทั้งสองค่า; 0 = ใช้งานไม่ได้, 1 = ถูกต้องสมบูรณ์)
#     3) noisy_text       : ข้อความจำลองผล OCR จากเอกสารแฟกซ์คุณภาพต่ำ (มีตัวอักษรปนในตัวเลข)
#     4) fix_ocr_numbers  : ฟังก์ชันแก้ตัวอักษรที่หน้าตาคล้ายตัวเลข (เช่น l → 1) เฉพาะในคำที่เป็นตัวเลข
#                           (จะใช้งานในแบบฝึกหัดที่ 5)
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ค่า precision, recall, F1 ของภาพคมชัด เทียบกับข้อความที่มีสัญญาณรบกวน
#   และตารางผลการตรวจที่สกัดได้จากข้อความที่มีสัญญาณรบกวน (ซึ่งเหลือเพียงแถวเดียว)
#   หมายเหตุ: ใช้ฟังก์ชัน extract_labs และรายการ KNOWN_TESTS จากขั้นตอนที่ 3
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: เฉลยของรายงาน
# ----------------------------------------------------------------------------
# สร้างตัวแปร ANSWER_LABS เป็นรายการ (list) ของคู่ค่า (tuple) ในรูป (ชื่อการตรวจ, ผลการตรวจ)
#   - [ ] คือรายการ, ( ) แต่ละชุดคือคู่ค่า 1 คู่ คั่นด้วย ,
#   - ค่าเหล่านี้ตรงกับรายงานต้นฉบับในขั้นตอนที่ 1 ใช้เป็น "คำตอบที่ถูกต้อง" สำหรับให้คะแนน
ANSWER_LABS = [("Hemoglobin", 11.2), ("Glucose (fasting)", 142), ("Creatinine", 1.8), ("LDL cholesterol", 165)]

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ฟังก์ชัน pair_f1 คำนวณ precision, recall และ F1
#   - true_pairs  : พารามิเตอร์ที่ 1 = รายการคู่ที่ถูกต้อง (เฉลย)
#   - found_pairs : พารามิเตอร์ที่ 2 = รายการคู่ที่ระบบสกัดได้
#   - ผลลัพธ์ (return) : ชุดค่า 3 ค่า (precision, recall, F1) ปัดเป็นทศนิยม 2 ตำแหน่ง
#   หลักการ: คู่ข้อมูลนับว่าถูกต้องเมื่อทั้งชื่อ "และ" ตัวเลขตรงกันเท่านั้น
# ----------------------------------------------------------------------------
def pair_f1(true_pairs, found_pairs):
    """คำนวณ precision, recall และ F1 ของรายการคู่ (ชื่อ, ค่า) โดยไม่คำนึงถึงตัวพิมพ์เล็กและใหญ่ของชื่อ"""
    # เตรียมเฉลยให้อยู่ในรูปแบบมาตรฐานเดียวกัน แล้วเก็บในตัวแปร true (list comprehension)
    #   - for n, v in true_pairs : วนหยิบทีละคู่ แยกเป็นชื่อ n และค่า v
    #   - n.lower()              : แปลงชื่อเป็นตัวพิมพ์เล็ก เพื่อไม่ให้ตัวพิมพ์เล็ก/ใหญ่มีผลต่อการเทียบ
    #   - float(v)               : แปลงค่าเป็นตัวเลขทศนิยม (142 กับ 142.0 จะถือว่าเท่ากัน)
    #   - round(..., 3)          : ปัดเศษเป็นทศนิยม 3 ตำแหน่ง ป้องกันความคลาดเคลื่อนเล็กน้อยของเลขทศนิยมในคอมพิวเตอร์
    true = [(n.lower(), round(float(v), 3)) for n, v in true_pairs]
    # เตรียมคู่ที่ระบบสกัดได้ในรูปแบบเดียวกัน แล้วเก็บในตัวแปร found
    found = [(n.lower(), round(float(v), 3)) for n, v in found_pairs]
    # tp (true positive — ผลบวกจริง) : ตัวนับจำนวนคู่ที่สกัดได้ถูกต้อง เริ่มต้นที่ 0
    tp = 0
    # วนลูปตรวจคู่ที่สกัดได้ทีละคู่ (เก็บในตัวแปร p)
    for p in found:
        # ถ้าคู่ p มีอยู่ในเฉลย (in = "อยู่ใน")
        if p in true:
            # 2 คำสั่งคั่นด้วย ;
            #   1) true.remove(p) : ลบคู่นั้นออกจากรายการเฉลย เพื่อไม่ให้นับซ้ำหากสกัดซ้ำ 2 ครั้ง
            #   2) tp += 1        : เพิ่มตัวนับคู่ที่ถูกต้องขึ้น 1
            #   (หลังจบลูป สิ่งที่เหลือใน true คือค่าในเฉลยที่ระบบหาไม่พบ = false negative)
            true.remove(p); tp += 1
    # คำนวณ precision = จำนวนที่ถูก ÷ จำนวนที่ระบบสกัดได้ทั้งหมด
    #   - len(found) : จำนวนคู่ที่สกัดได้
    #   - รูปแบบ "A if เงื่อนไข else B" : ถ้า found มีข้อมูล (ไม่ว่าง) ให้คำนวณ A มิฉะนั้นให้เป็น 0.0
    #     (ป้องกันการหารด้วยศูนย์เมื่อระบบสกัดไม่ได้อะไรเลย)
    precision = tp / len(found) if found else 0.0
    # คำนวณ recall = จำนวนที่ถูก ÷ จำนวนค่าในเฉลยทั้งหมด
    #   - tp + len(true) : ที่ถูก + ที่ยังเหลือในเฉลย (หาไม่พบ) = จำนวนในเฉลยทั้งหมด
    #   - ถ้าผลรวมเป็น 0 ให้ recall เป็น 0.0 (ป้องกันการหารด้วยศูนย์)
    recall = tp / (tp + len(true)) if (tp + len(true)) else 0.0
    # คำนวณ F1 = 2 × precision × recall ÷ (precision + recall) คือค่าเฉลี่ยฮาร์มอนิก
    #   - ถ้า precision + recall เป็น 0 ให้ F1 เป็น 0.0 (ป้องกันการหารด้วยศูนย์)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    # ส่งค่าทั้งสามกลับ โดยปัดเศษเป็นทศนิยม 2 ตำแหน่งด้วย round(ค่า, 2)
    #   การเขียนค่าคั่นด้วย , หลัง return จะส่งกลับเป็นชุดค่า (tuple) 3 ค่า
    return round(precision, 2), round(recall, 2), round(f1, 2)

# ----------------------------------------------------------------------------
# ส่วนที่ 3: ฟังก์ชัน lab_pairs แปลงตารางผลการตรวจเป็นรายการคู่ (ชื่อ, ค่า)
#   - df : พารามิเตอร์ คือตาราง (DataFrame) ที่ได้จาก extract_labs
# ----------------------------------------------------------------------------
def lab_pairs(df):
    # จับคู่ค่าจาก 2 คอลัมน์ทีละแถว แล้วส่งกลับเป็นรายการ
    #   - df["test"], df["result"] : ดึงคอลัมน์ชื่อการตรวจ และคอลัมน์ผลการตรวจ
    #   - zip(...)  : ฟังก์ชันของ Python ที่จับคู่สมาชิกลำดับเดียวกันจากหลายชุดข้อมูล
    #                 เช่น ("Hemoglobin", 11.2), ("Creatinine", 1.8), ...
    #   - list(...) : แปลงผลของ zip ให้เป็นรายการ (list) ที่นำไปใช้ต่อได้
    return list(zip(df["test"], df["result"]))

# ----------------------------------------------------------------------------
# ส่วนที่ 4: ข้อความจำลองผล OCR ที่มีสัญญาณรบกวน (noise)
# ----------------------------------------------------------------------------
# ตัวอย่างผล OCR ของรายงานฉบับเดิมที่สแกนด้วยคุณภาพต่ำ
# สร้างตัวแปร noisy_text เก็บข้อความหลายบรรทัด (ใช้เครื่องหมาย """ ... """) ซึ่งเลียนแบบความผิดพลาดของ OCR
#   - l42 : ตัวอักษรแอลตัวเล็ก (l) แทนเลข 1 ในค่า glucose
#   - 1.B : ตัวอักษรบีตัวใหญ่ (B) ปนอยู่ในค่า creatinine
#   - I65 : ตัวอักษรไอตัวใหญ่ (I) แทนเลข 1 ในค่า LDL
#   - 5O0 : ตัวอักษรโอตัวใหญ่ (O) แทนเลข 0 ในขนาดยา Metformin
#   - นอกจากนี้ ช่องว่างระหว่างคอลัมน์ถูกยุบเหลือช่องเดียว และบรรทัดว่างหายไป
noisy_text = """CENTRAL HOSPITAL - LABORATORY REPORT
Patient: John Doe MRN: 001234 Age: 57 Sex: M
Date: 2026-10-17
TEST RESULT UNIT REFERENCE
Hemoglobin 11.2 g/dL 13.5-17.5
Glucose (fasting) l42 mg/dL 70-99
Creatinine 1.B mg/dL 0.7-1.3
LDL cholesterol I65 mg/dL <100
Medications: Metformin 5O0 mg BID; Atorvastatin 20 mg OD
Impression: Anemia, hyperglycemia, elevated LDL."""

# ----------------------------------------------------------------------------
# ส่วนที่ 5: ฟังก์ชัน fix_ocr_numbers แก้ตัวอักษรที่หน้าตาคล้ายตัวเลข
#   - text  : พารามิเตอร์ที่ 1 = ข้อความที่ต้องการแก้
#   - fixes : พารามิเตอร์ที่ 2 = dict กำหนดการแทนที่ ในรูป {"ตัวอักษร": "ตัวเลข"} เช่น {"l": "1"}
#   - ผลลัพธ์ (return) : ข้อความที่แก้ไขแล้ว
#   แก้ "เฉพาะคำที่มีลักษณะเป็นตัวเลข" (มีเลขจริงอย่างน้อย 1 หลัก) คำอย่าง Hemoglobin จึงไม่ถูกเปลี่ยน
# ----------------------------------------------------------------------------
def fix_ocr_numbers(text, fixes):
    """แทนตัวอักษรที่มีรูปร่างคล้ายตัวเลขด้วยตัวเลข เฉพาะในคำที่มีลักษณะเป็นตัวเลข (เช่น 'l42', '5O0')"""
    # รวบรวมตัวอักษรทั้งหมดที่ต้องแก้มาต่อเป็นข้อความเดียว แล้วเก็บในตัวแปร chars
    #   - "".join(fixes) : การวนผ่าน dict จะได้ "คีย์" ทีละตัว จึงได้ตัวอักษรที่ต้องแก้มาต่อกัน เช่น "OolI"
    #   - re.escape(...) : ใส่ \ หน้าสัญลักษณ์พิเศษ (ถ้ามี) เพื่อให้ regex ถือเป็นตัวอักษรธรรมดา
    chars = re.escape("".join(fixes))
    # สร้างรูปแบบการค้นหา "คำที่มีลักษณะเป็นตัวเลข" แล้วเก็บในตัวแปร token
    #   - rf"..." : raw string + f-string (แทรกค่า chars ลงในตำแหน่ง {chars})
    #   ความหมายของรูปแบบทีละส่วน:
    #     (?<![A-Za-z0-9])   : "มองย้อนหลังแบบปฏิเสธ" (negative lookbehind) = อักขระก่อนหน้าต้อง "ไม่ใช่"
    #                          ตัวอักษรอังกฤษหรือตัวเลข คือคำต้องเริ่มที่ขอบคำ
    #     (                  : เริ่มกลุ่มที่ 1 (ส่วนที่จะถูกแก้ไข)
    #       [0-9.{chars}]*   : อักขระที่เป็นตัวเลข 0-9, จุด . หรือตัวอักษรที่ต้องแก้ จำนวนกี่ตัวก็ได้ (รวม 0 ตัว)
    #                          (ภายใน [ ] จุด . หมายถึงจุดจริง ๆ)
    #       [0-9]            : ต้องมีตัวเลขจริงอย่างน้อย 1 หลัก (เพื่อยืนยันว่าเป็นคำแบบตัวเลข)
    #       [0-9.{chars}]*   : ตามด้วยอักขระชุดเดียวกันอีกกี่ตัวก็ได้
    #     )                  : จบกลุ่มที่ 1
    #     (?![A-Za-z0-9])    : "มองไปข้างหน้าแบบปฏิเสธ" = อักขระถัดไปต้องไม่ใช่ตัวอักษรอังกฤษหรือตัวเลข (จบที่ขอบคำ)
    #   ตัวอย่าง: "l42" และ "5O0" ตรงรูปแบบ แต่ "Hemoglobin" ไม่ตรง เพราะไม่มีตัวเลขจริงเลย
    token = re.compile(rf"(?<![A-Za-z0-9])([0-9.{chars}]*[0-9][0-9.{chars}]*)(?![A-Za-z0-9])")
    # แทนที่ทุกคำที่ตรงรูปแบบด้วยคำที่แก้ไขแล้ว และส่งข้อความผลลัพธ์กลับ
    #   - token.sub(ตัวแทนที่, text) : ค้นหาทุกตำแหน่งใน text ที่ตรงรูปแบบ แล้วแทนที่
    #     ในที่นี้ "ตัวแทนที่" เป็นฟังก์ชัน จึงคำนวณข้อความใหม่แยกให้แต่ละคำที่พบ
    #   - lambda m: ... : การสร้างฟังก์ชันสั้น ๆ แบบไม่ตั้งชื่อในบรรทัดเดียว
    #                     m คือผลการค้นพบแต่ละครั้ง (match) ส่วนหลัง : คือค่าที่ส่งกลับ
    #   - m.group(1)            : ข้อความของกลุ่มที่ 1 ที่ค้นพบ เช่น "l42"
    #   - for c in m.group(1)   : วนหยิบทีละอักขระ c
    #   - fixes.get(c, c)       : ค้นใน dict ถ้ามีคีย์ c ให้ใช้ค่าที่กำหนด (เช่น "l" → "1")
    #                             ถ้าไม่มี ให้ใช้ c เดิม (อาร์กิวเมนต์ที่ 2 คือค่าที่ใช้เมื่อหาไม่พบ)
    #   - "".join(...)          : นำอักขระที่แก้แล้วมาต่อกันเป็นคำใหม่ เช่น "142"
    return token.sub(lambda m: "".join(fixes.get(c, c) for c in m.group(1)), text)

# ----------------------------------------------------------------------------
# ส่วนที่ 6: เปรียบเทียบคะแนนระหว่างข้อความจากภาพคมชัด กับข้อความที่มีสัญญาณรบกวน
# ----------------------------------------------------------------------------
# คำนวณและแสดงคะแนนของข้อความจากภาพคมชัด (ocr_text) ทำงานจากในออกนอก
#   1) extract_labs(ocr_text) : สกัดตารางผลการตรวจจากข้อความ OCR
#   2) lab_pairs(...)         : แปลงตารางเป็นรายการคู่ (ชื่อ, ค่า)
#   3) pair_f1(ANSWER_LABS, ...) : เทียบกับเฉลย ได้ (precision, recall, F1)
#   4) print(ข้อความ, ผลลัพธ์) : แสดงข้อความนำ ตามด้วยค่าทั้งสาม
print("OCR ภาพคมชัด         -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(ocr_text))))
# ทำแบบเดียวกันกับข้อความที่มีสัญญาณรบกวน (noisy_text) เพื่อเปรียบเทียบว่าคะแนนลดลงเท่าใด
print("OCR มีสัญญาณรบกวน    -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(noisy_text))))
# สกัดตารางผลการตรวจจาก noisy_text เป็นบรรทัดสุดท้ายของเซลล์ Jupyter/Colab จึงแสดงตารางให้อัตโนมัติ
#   ผู้เรียนจะเห็นว่าเหลือเพียงแถวเดียว เพราะค่าที่มีตัวอักษรปนไม่ตรงกับรูปแบบตัวเลข
extract_labs(noisy_text)

💡 **คำอธิบายผลลัพธ์**

ผลการตรวจเหลือเพียงแถวเดียว เนื่องจาก `l42` (ตัวอักษรแอล) `I65` (ตัวอักษรไอ) และ `1.B` (ตัวอักษรบี) ไม่ใช่ตัวเลข รูปแบบการค้นหาจึงข้ามบรรทัดเหล่านี้ ฟังก์ชัน `fix_ocr_numbers` แก้ไขตัวอักษรที่มีรูปร่างคล้ายตัวเลข **เฉพาะในคำที่มีลักษณะเป็นตัวเลข** ดังนั้นคำอย่าง "Hemoglobin" ซึ่งมีตัวอักษร `l` จะไม่ถูกแก้ไข


In [ ]:
# ⭐ Exercise 5 — แบบฝึกหัดที่ 5: การระบุและแก้ไขข้อผิดพลาดจาก OCR
# OCR_FIXES กำหนดว่าตัวอักษรที่มีรูปร่างคล้ายตัวเลขแต่ละตัวควรแทนด้วยตัวเลขใด
# ➜ ให้รันเซลล์และพิจารณาข้อความที่แสดง ยังมีผลการตรวจ 1 ค่าที่อ่านผิดและยังไม่ได้รับการแก้ไข (F1 < 1)
#   ให้ระบุค่าดังกล่าว แล้วเพิ่มรายการ  "ตัวอักษร": "ตัวเลข"  อีก 1 รายการใน OCR_FIXES จนค่า F1 เท่ากับ 1.0
#   (ค่า F1 = 1.0 ต้องมีชื่อการตรวจครบทั้ง 4 รายการจากแบบฝึกหัดที่ 2)
# คำแนะนำ: พิจารณาบรรทัด Creatinine ว่าตัวอักษรที่ปรากฏมีรูปร่างคล้ายตัวเลขใด
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ทำความสะอาดข้อความ OCR ที่มีสัญญาณรบกวน (noisy_text) ด้วยกฎแทนที่ตัวอักษรที่หน้าตาคล้ายตัวเลข
#   แล้ววัดว่าการสกัดผลการตรวจดีขึ้นเพียงใด (ดูค่า precision, recall และ F1 ก่อน/หลังแก้ไข)
#   ข้อมูลนำเข้า : noisy_text และกฎการแก้ไข OCR_FIXES
#   ผลลัพธ์      : ข้อความหลังแก้ไข, คะแนนก่อนและหลังแก้ไข, และตารางผลการตรวจที่สกัดได้
#   ประเด็นทางคลินิก: อักขระที่อ่านผิดเพียงตัวเดียวอาจทำให้ค่าผิดปกติหายไปจากตาราง
#   สิ่งที่ผู้เรียนต้องทำ: หาค่าที่ยังอ่านผิด แล้วเพิ่มกฎอีก 1 รายการใน OCR_FIXES จน F1 = 1.0
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: กฎการแก้ไขตัวอักษรที่ OCR อ่านผิด
# ----------------------------------------------------------------------------
# สร้างตัวแปร OCR_FIXES เป็นพจนานุกรม (dict) ในรูป {"ตัวอักษรที่อ่านผิด": "ตัวเลขที่ถูกต้อง", ...}
#   - "O": "0" : อักษรโอตัวใหญ่ → เลขศูนย์
#   - "o": "0" : อักษรโอตัวเล็ก → เลขศูนย์
#   - "l": "1" : อักษรแอลตัวเล็ก → เลขหนึ่ง
#   - "I": "1" : อักษรไอตัวใหญ่ → เลขหนึ่ง
#   - # TODO ท้ายบรรทัด : ผู้เรียนต้องเพิ่มกฎอีก 1 คู่ในรูปแบบเดียวกัน (คั่นด้วย , ก่อนปิดวงเล็บ })
#     โดยดูจากบรรทัด Creatinine ในข้อความที่แสดงผล
OCR_FIXES = {"O": "0", "o": "0", "l": "1", "I": "1"}   # TODO เพิ่มอีก 1 รายการ

# ----------------------------------------------------------------------------
# ส่วนที่ 2: แก้ข้อความด้วย OCR_FIXES แล้วเปรียบเทียบคะแนนก่อนและหลังแก้
# ----------------------------------------------------------------------------
# เรียกฟังก์ชัน fix_ocr_numbers (สร้างไว้ในขั้นตอนที่ 6) เพื่อแก้ข้อความที่มีสัญญาณรบกวน แล้วเก็บผลใน fixed_text
#   - noisy_text : ข้อความ OCR คุณภาพต่ำที่ต้องการแก้
#   - OCR_FIXES  : กฎการแทนที่ตัวอักษร → ตัวเลข (ใช้เฉพาะในคำที่มีลักษณะเป็นตัวเลข)
fixed_text = fix_ocr_numbers(noisy_text, OCR_FIXES)
# แสดงข้อความหลังแก้ไข เพื่อให้ผู้เรียนตรวจดูว่าค่าใดยังผิดอยู่
#   - fixed_text : ข้อความที่แก้แล้ว
#   - "\n"       : ข้อความขึ้นบรรทัดใหม่ เพิ่มบรรทัดว่างคั่นก่อนผลลัพธ์ถัดไป
print(fixed_text, "\n")
# แสดงคะแนนของข้อความ "ก่อน" แก้ไข (ใช้เป็นจุดเปรียบเทียบ) ทำงานจากในออกนอก
#   1) extract_labs(noisy_text)   : สกัดตารางผลการตรวจ
#   2) lab_pairs(...)             : แปลงเป็นรายการคู่ (ชื่อ, ค่า)
#   3) pair_f1(ANSWER_LABS, ...)  : เทียบกับเฉลย ได้ (precision, recall, F1)
print("OCR มีสัญญาณรบกวน      -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(noisy_text))))
# แสดงคะแนนของข้อความ "หลัง" แก้ไขด้วย OCR_FIXES (ขั้นตอนเหมือนบรรทัดก่อนหน้า แต่ใช้ fixed_text)
print("หลังแก้ไขด้วย OCR_FIXES -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(fixed_text))))
# สกัดตารางผลการตรวจจากข้อความที่แก้แล้ว เป็นบรรทัดสุดท้ายของเซลล์ Jupyter/Colab จึงแสดงตารางให้อัตโนมัติ
extract_labs(fixed_text)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 5)
#   ทำความสะอาดข้อความ OCR ที่มีสัญญาณรบกวน (noisy_text) ด้วยกฎแทนที่ตัวอักษรที่หน้าตาคล้ายตัวเลข
#   แล้ววัดว่าการสกัดผลการตรวจดีขึ้นเพียงใด (ดูค่า precision, recall และ F1 ก่อน/หลังแก้ไข)
#   คำตอบของแบบฝึกหัด: ค่า creatinine ถูกอ่านเป็น "1.B" จึงต้องเพิ่มกฎ "B": "8"
#   ทำให้กู้ค่า 1.8 mg/dL (ซึ่งผิดปกติ) กลับมาได้ และ F1 เพิ่มเป็น 1.0
#   ข้อควรระวัง: กฎนี้อาจเปลี่ยนตัว B ที่ถูกต้องในคำที่มีตัวเลขปนเป็น 8 ได้ ค่าที่สกัดได้จึงควรผ่าน
#   การตรวจสอบความสมเหตุสมผล (plausibility check) และการทบทวนโดยบุคลากรก่อนใช้ทางคลินิก
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: กฎการแก้ไขตัวอักษรที่ OCR อ่านผิด (ครบตามเฉลย)
# ----------------------------------------------------------------------------
# สร้างตัวแปร OCR_FIXES เป็นพจนานุกรม (dict) ในรูป {"ตัวอักษรที่อ่านผิด": "ตัวเลขที่ถูกต้อง", ...}
#   - "O": "0" : อักษรโอตัวใหญ่ → เลขศูนย์
#   - "o": "0" : อักษรโอตัวเล็ก → เลขศูนย์
#   - "l": "1" : อักษรแอลตัวเล็ก → เลขหนึ่ง
#   - "I": "1" : อักษรไอตัวใหญ่ → เลขหนึ่ง
#   - "B": "8" : อักษรบีตัวใหญ่ → เลขแปด (กฎที่เพิ่มตามเฉลย เพื่อแก้ "1.B" เป็น "1.8")
OCR_FIXES = {"O": "0", "o": "0", "l": "1", "I": "1", "B": "8"}

# ----------------------------------------------------------------------------
# ส่วนที่ 2: แก้ข้อความด้วย OCR_FIXES แล้วเปรียบเทียบคะแนนก่อนและหลังแก้
# ----------------------------------------------------------------------------
# เรียกฟังก์ชัน fix_ocr_numbers (สร้างไว้ในขั้นตอนที่ 6) เพื่อแก้ข้อความที่มีสัญญาณรบกวน แล้วเก็บผลใน fixed_text
#   - noisy_text : ข้อความ OCR คุณภาพต่ำที่ต้องการแก้
#   - OCR_FIXES  : กฎการแทนที่ตัวอักษร → ตัวเลข (ใช้เฉพาะในคำที่มีลักษณะเป็นตัวเลข)
fixed_text = fix_ocr_numbers(noisy_text, OCR_FIXES)
# แสดงข้อความหลังแก้ไข เพื่อให้ผู้เรียนตรวจดูว่าค่าใดยังผิดอยู่
#   - fixed_text : ข้อความที่แก้แล้ว
#   - "\n"       : ข้อความขึ้นบรรทัดใหม่ เพิ่มบรรทัดว่างคั่นก่อนผลลัพธ์ถัดไป
print(fixed_text, "\n")
# แสดงคะแนนของข้อความ "ก่อน" แก้ไข (ใช้เป็นจุดเปรียบเทียบ) ทำงานจากในออกนอก
#   1) extract_labs(noisy_text)   : สกัดตารางผลการตรวจ
#   2) lab_pairs(...)             : แปลงเป็นรายการคู่ (ชื่อ, ค่า)
#   3) pair_f1(ANSWER_LABS, ...)  : เทียบกับเฉลย ได้ (precision, recall, F1)
print("OCR มีสัญญาณรบกวน      -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(noisy_text))))
# แสดงคะแนนของข้อความ "หลัง" แก้ไขด้วย OCR_FIXES (ขั้นตอนเหมือนบรรทัดก่อนหน้า แต่ใช้ fixed_text)
print("หลังแก้ไขด้วย OCR_FIXES -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(fixed_text))))
# สกัดตารางผลการตรวจจากข้อความที่แก้แล้ว เป็นบรรทัดสุดท้ายของเซลล์ Jupyter/Colab จึงแสดงตารางให้อัตโนมัติ
extract_labs(fixed_text)

💡 **คำอธิบายผลลัพธ์**

อักขระที่อ่านผิดเพียงตัวเดียวทำให้ค่า creatinine **1.8 mg/dL** (ซึ่งผิดปกติ) หายไปจากตาราง กฎการทำความสะอาดข้อมูลอย่างง่ายช่วยกู้ค่าดังกล่าวกลับมา และทำให้ค่า F1 เพิ่มเป็น 1.0 อย่างไรก็ตาม กฎการทำความสะอาดข้อมูลอาจ *ก่อให้เกิด* ข้อผิดพลาดได้เช่นกัน (ตัวอักษร `B` ที่ถูกต้องอาจถูกเปลี่ยนเป็น `8`) ค่าที่สกัดได้จึงควรผ่านการตรวจสอบความสมเหตุสมผล (plausibility check) เช่น ค่า creatinine อยู่ในช่วงที่เป็นไปได้ทางสรีรวิทยาหรือไม่ และควรได้รับการทบทวนโดยบุคลากรก่อนนำไปใช้ทางคลินิก


✍️ **คำถามที่ 3** หาก OCR อ่านจุดทศนิยมตกหล่น ทำให้ค่า hemoglobin `11.2` กลายเป็น `112` g/dL หรือค่า creatinine `1.8` สูญหายไป จงอธิบายผลกระทบที่อาจเกิดขึ้นต่อผู้ป่วย และเสนอ **มาตรการป้องกัน 2 ประการ** ที่โรงพยาบาลควรกำหนดเมื่อใช้ระบบสกัดสารสนเทศด้วย OCR แบบอัตโนมัติ


✍️ **คำตอบ:**



## ส่วนเสริม (ไม่บังคับ) — การสกัดสารสนเทศด้วยโมเดล NLP ที่เรียนรู้จากข้อมูล (NER)

กฎที่เขียนขึ้นเองมีข้อจำกัด เนื่องจากต้องเพิ่มกฎใหม่ทุกครั้งที่พบตัวสะกด หน่วย หรือรูปแบบเอกสารที่แตกต่างไป ระบบในปัจจุบันจึงใช้โมเดล **NER** (*Named-Entity Recognition* — การรู้จำชื่อเฉพาะ) ร่วมด้วย โมเดลประเภทนี้เรียนรู้จากตัวอย่างที่มีการกำกับป้าย (label) จำนวนมาก เพื่อระบุว่าคำใดเป็น *ยา* *ขนาดยา* *ผลการตรวจ* หรือ *โรค* แม้ในประโยคที่ไม่เคยพบมาก่อน ผู้เรียนที่สนใจสามารถทดลองใช้ [scispaCy](https://allenai.github.io/scispacy/) หรือโมเดล NER ทางชีวการแพทย์บน Hugging Face (ขนาดดาวน์โหลดประมาณ 250 MB; ใช้ CPU ได้) โดยคัดลอกโค้ดด้านล่างไปวางในเซลล์โค้ดใหม่ (**+ Code**)

```python
!pip -q install transformers
from transformers import pipeline
ner = pipeline("token-classification", model="d4data/biomedical-ner-all", aggregation_strategy="simple")
ner("Metformin 500 mg BID for type 2 diabetes; hemoglobin 11.2 g/dL")
```

ผลลัพธ์แต่ละรายการประกอบด้วย `entity_group` (เช่น `Medication`, `Dosage`, `Lab_value`) และคำที่ตรวจพบใน `word` ทั้งนี้ โมเดล NER อาจผิดพลาดได้ และยังต้องอาศัยข้อความจาก OCR ที่มีคุณภาพเพียงพอ


✍️ **คำถามที่ 4 (คำถามสะท้อนคิดหลัก)** จงอธิบายโดยสังเขป (2–4 ประโยค) ว่าในปฏิบัติการนี้ OCR และกฎที่เขียนขึ้นเองมีข้อจำกัดในประเด็นใด (เช่น หน่วย รูปแบบเอกสาร อักขระที่มีรูปร่างคล้ายกัน ตัวสะกด หรือลายมือ) และโมเดล NER ที่เรียนรู้จากข้อมูล หรือขั้นตอนการทบทวนโดยบุคลากร (human-in-the-loop) ช่วยลดข้อจำกัดดังกล่าวได้อย่างไร


✍️ **คำตอบ:**



## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

> **ส่วนนี้ไม่บังคับ และไม่นับเป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง** ผู้เรียนที่ไม่เข้าร่วมสามารถข้ามไปยังหัวข้อ *ชิ้นงานที่ต้องส่ง*

ให้ทำ OCR กับ **ภาพรายงานผลการตรวจทางห้องปฏิบัติการสังเคราะห์ 20 ฉบับ** ของรายวิชา และสกัด (1) การตรวจทุกรายการพร้อมผลการตรวจที่เป็นตัวเลข และ (2) ยาทุกรายการพร้อมขนาดยาในหน่วย mg **ตัวชี้วัด: ค่าเฉลี่ยของ F1 ของคู่ (ชื่อการตรวจ, ผลการตรวจ) และ F1 ของคู่ (ชื่อยา, ขนาดยา)** (ชื่อการตรวจเปรียบเทียบหลังแปลงเป็นตัวพิมพ์เล็กและตัดส่วนตั้งแต่เครื่องหมาย `(` ออก ส่วนชื่อยาเปรียบเทียบด้วยชื่อสามัญทางยา)

รายงานชุดทดสอบมี **รายการตรวจและหน่วยที่หลากหลายกว่า** ตัวอย่างข้างต้น (`%`, `mmol/L`, `U/L`, `10^3/uL` เป็นต้น) และบางฉบับมีภาพเอียงเล็กน้อยหรือมีสัญญาณรบกวน โค้ดตั้งต้น (baseline) ด้านล่างสามารถรันได้ทันที และสร้างไฟล์ `lab07_submission.json` ในรูปแบบที่ถูกต้อง

`{"R01": {"labs": [{"test": "Glucose (fasting)", "result": 95, "unit": "mg/dL"}, ...], "medications": [{"name": "Metformin", "dose_mg": 850}, ...]}, ...}`

การส่งผลลัพธ์ ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์ ⚙️ การตั้งค่า (ผู้สอนประกาศรหัสเข้าร่วมในการเรียนการสอนผ่าน Zoom) แล้วรันเซลล์ดังกล่าวอีกครั้ง

**หลักความซื่อสัตย์ทางวิชาการ (honor code):** ให้ส่งเฉพาะผลการทำนายที่ได้จากโปรแกรมของผู้เรียนเองซึ่งประมวลผลจากภาพ ห้ามพิมพ์หรือแก้ไขผลของรายงานชุดทดสอบด้วยตนเอง


In [ ]:
# 🏆 Baseline — ทำ OCR และสกัดสารสนเทศจากรายงานทั้ง 20 ฉบับ สร้างไฟล์ lab07_submission.json และส่งผลลัพธ์
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (โจทย์ท้าทาย — ไม่บังคับ)
#   โค้ดตั้งต้น (baseline) สำหรับตารางอันดับ ทำงานครบทั้งกระบวนการ (pipeline) กับรายงานสังเคราะห์ 20 ฉบับ
#     1) ดาวน์โหลดรายชื่อไฟล์ภาพรายงาน (manifest) และภาพรายงานแต่ละฉบับ
#     2) ทำ OCR แปลงภาพเป็นข้อความ
#     3) แก้ตัวอักษรที่หน้าตาคล้ายตัวเลข (ใช้ OCR_FIXES จากแบบฝึกหัดที่ 5)
#     4) สกัดผลการตรวจ (ชื่อ ผล หน่วย) และรายการยา (ชื่อ ขนาดยา mg) ด้วยรูปแบบการค้นหา
#     5) บันทึกผลทั้งหมดเป็นไฟล์ lab07_submission.json และส่งไปยังตารางอันดับ (ถ้ากรอก NAME/JOIN_CODE ไว้)
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ผลการสกัดของรายงาน R01 ในรูปแบบ JSON, ข้อความยืนยันการสร้างไฟล์
#   และคะแนน/อันดับ (หากส่งผล) หรือข้อความแนะนำวิธีส่งผล
#   หมายเหตุ: ต้องรันเซลล์ก่อนหน้าทั้งหมด (รวมถึงเฉลยแบบฝึกหัดที่ 5) ก่อน เพราะใช้ fix_ocr_numbers และ OCR_FIXES
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: นำเข้าไลบรารีและโหลดรายชื่อไฟล์รายงาน
# ----------------------------------------------------------------------------
# นำเข้าไลบรารี json (ไลบรารีมาตรฐานของ Python) สำหรับอ่าน/เขียนข้อมูลในรูปแบบ JSON
#   (JSON คือรูปแบบไฟล์ข้อความที่เก็บข้อมูลเป็นโครงสร้าง คล้าย dict และ list ของ Python)
import json

# โหลดตารางรายชื่อรายงาน แล้วเก็บไว้ในตัวแปร manifest (ทำงานจากในออกนอก)
#   1) lab_data("ocr/manifest.csv") : ฟังก์ชันจากเซลล์การตั้งค่า ดาวน์โหลดไฟล์ manifest.csv
#                                     ในโฟลเดอร์ ocr ของคลังข้อมูลรายวิชา แล้วคืนตำแหน่งไฟล์ในเครื่อง
#   2) pd.read_csv(...)             : ฟังก์ชันของ pandas อ่านไฟล์ CSV (ตารางที่คั่นด้วยจุลภาค) เป็น DataFrame
#   ตารางนี้มีคอลัมน์ id (รหัสรายงาน เช่น R01) และ file (ชื่อไฟล์ภาพของรายงานนั้น)
manifest = pd.read_csv(lab_data("ocr/manifest.csv"))

# ----------------------------------------------------------------------------
# ส่วนที่ 2: รูปแบบการค้นหาผลการตรวจและรายการยา
# ----------------------------------------------------------------------------
# รูปแบบการค้นหาผลการตรวจแบบทั่วไป (ไม่จำกัดเฉพาะชื่อในรายการที่กำหนด): บรรทัดที่ขึ้นต้นด้วยชื่อการตรวจ
# (ตัวอักษร/ตัวเลข/ช่องว่าง/()/-) ตามด้วยตัวเลข และตามด้วยหน่วย
# สร้างรูปแบบการค้นหาผลการตรวจ เก็บในตัวแปร LAB_RE
#   - re.compile(รูปแบบ, re.M) : แปลงรูปแบบให้พร้อมใช้; re.M ทำให้ ^ หมายถึงต้นบรรทัดทุกบรรทัด
#   ความหมายของรูปแบบทีละส่วน:
#     ^                    : ต้องเริ่มที่ต้นบรรทัด
#     (                    : เริ่มกลุ่มที่ 1 = ชื่อการตรวจ
#       [A-Za-z]           : ตัวแรกต้องเป็นตัวอักษรอังกฤษ (พิมพ์เล็กหรือใหญ่)
#       [A-Za-z0-9 ()/-]*? : ตามด้วยตัวอักษร ตัวเลข ช่องว่าง วงเล็บ ( ) เครื่องหมาย / หรือ - กี่ตัวก็ได้
#                            *? = แบบ "น้อยที่สุดเท่าที่ทำได้" (lazy) เพื่อให้ชื่อหยุดก่อนถึงตัวเลขผลการตรวจตัวแรก
#     )                    : จบกลุ่มที่ 1
#     \s+                  : ตามด้วยช่องว่างอย่างน้อย 1 ช่อง
#     (\d+(?:\.\d+)?)      : กลุ่มที่ 2 = ผลการตรวจ ตัวเลขที่อาจมีทศนิยม เช่น 95 หรือ 1.8
#     \s+                  : ตามด้วยช่องว่างอย่างน้อย 1 ช่อง
#     (\S+)                : กลุ่มที่ 3 = หน่วย คืออักขระที่ไม่ใช่ช่องว่างติดกัน เช่น mg/dL, %, 10^3/uL
LAB_RE = re.compile(r"^([A-Za-z][A-Za-z0-9 ()/-]*?)\s+(\d+(?:\.\d+)?)\s+(\S+)", re.M)
# สร้างรูปแบบการค้นหายา เก็บในตัวแปร MED_RE (เหมือนกับขั้นตอนที่ 5)
#   ([A-Z][a-z]+)     : กลุ่มที่ 1 = ชื่อยา ขึ้นต้นตัวพิมพ์ใหญ่ ตามด้วยตัวพิมพ์เล็ก
#   \s+               : ช่องว่างอย่างน้อย 1 ช่อง
#   (\d+(?:\.\d+)?)   : กลุ่มที่ 2 = ขนาดยา (ตัวเลขที่อาจมีทศนิยม)
#   \s*mg             : ช่องว่าง 0 ช่องขึ้นไป ตามด้วย mg
#   (?![A-Za-z/])     : หลัง mg ต้องไม่ใช่ตัวอักษรหรือ / (ตัด mg/dL ออก)
MED_RE = re.compile(r"([A-Z][a-z]+)\s+(\d+(?:\.\d+)?)\s*mg(?![A-Za-z/])")

# ----------------------------------------------------------------------------
# ส่วนที่ 3: ฟังก์ชัน ocr_image ทำ OCR กับไฟล์ภาพ 1 ไฟล์
#   - path : พารามิเตอร์ คือตำแหน่งไฟล์ภาพในเครื่อง
#   - ผลลัพธ์ (return) : ข้อความที่ OCR อ่านได้
# ----------------------------------------------------------------------------
def ocr_image(path):
    # เปิดภาพด้วย Image.open(path) แล้วส่งให้ pytesseract.image_to_string อ่านเป็นข้อความ และส่งกลับ
    #   - config="--psm 6" : โหมดแบ่งหน้าแบบที่ 6 = มองทั้งภาพเป็นข้อความกลุ่มเดียวหลายบรรทัด
    return pytesseract.image_to_string(Image.open(path), config="--psm 6")

# ----------------------------------------------------------------------------
# ส่วนที่ 4: ฟังก์ชัน extract_report สกัดผลการตรวจและรายการยาจากข้อความของรายงาน 1 ฉบับ
#   - text : พารามิเตอร์ คือข้อความ OCR ของรายงาน
#   - ผลลัพธ์ (return) : dict {"labs": [...], "medications": [...]} ตามรูปแบบที่ตารางอันดับกำหนด
# ----------------------------------------------------------------------------
def extract_report(text):
    # แก้ตัวอักษรที่หน้าตาคล้ายตัวเลขก่อน แล้วเก็บผลทับลงตัวแปร text เดิม
    #   - fix_ocr_numbers(...) : ฟังก์ชันจากขั้นตอนที่ 6
    #   - OCR_FIXES            : กฎการแทนที่จากแบบฝึกหัดที่ 5
    text = fix_ocr_numbers(text, OCR_FIXES)
    # สกัดผลการตรวจทั้งหมด สร้างเป็นรายการของ dict แล้วเก็บในตัวแปร labs (ตัวแปรภายในฟังก์ชันนี้เท่านั้น)
    #   - LAB_RE.findall(text) : ค้นหาทุกบรรทัดที่ตรงรูปแบบ ได้รายการของชุด (ชื่อ, ผล, หน่วย)
    #   - for t, v, u in ...   : วนหยิบทีละชุด แยกเป็น t (ชื่อ), v (ผล), u (หน่วย)
    #   - {"test": ..., "result": ..., "unit": ...} : สร้าง dict 1 รายการต่อการตรวจ 1 รายการ
    #     - t.strip() : ตัดช่องว่างหัวท้ายของชื่อการตรวจ
    #     - float(v)  : แปลงผลการตรวจจากข้อความเป็นตัวเลขทศนิยม
    #     - u         : หน่วย (ข้อความ)
    labs = [{"test": t.strip(), "result": float(v), "unit": u} for t, v, u in LAB_RE.findall(text)]
    # สกัดรายการยาทั้งหมดในลักษณะเดียวกัน เก็บในตัวแปร meds
    #   - MED_RE.findall(text) : ได้รายการของคู่ (ชื่อยา, ขนาดยา)
    #   - {"name": n, "dose_mg": float(d)} : dict 1 รายการต่อยา 1 ตัว โดยแปลงขนาดยาเป็นตัวเลข
    meds = [{"name": n, "dose_mg": float(d)} for n, d in MED_RE.findall(text)]
    # ส่งกลับ dict ที่มี 2 คีย์ : "labs" (รายการผลการตรวจ) และ "medications" (รายการยา)
    return {"labs": labs, "medications": meds}

# ----------------------------------------------------------------------------
# ส่วนที่ 5: วนประมวลผลรายงานทุกฉบับ
# ----------------------------------------------------------------------------
# สร้าง dict ว่าง ({ } ที่ไม่มีข้อมูล) ชื่อ submission สำหรับเก็บผลของรายงานทุกฉบับ ในรูป {รหัสรายงาน: ผลการสกัด}
submission = {}
# วนลูปทีละรายงาน
#   - zip(manifest["id"], manifest["file"]) : จับคู่รหัสรายงานกับชื่อไฟล์ภาพ แถวต่อแถว
#   - rid : รหัสรายงานในรอบนั้น (เช่น "R01")
#   - f   : ชื่อไฟล์ภาพของรายงานในรอบนั้น
for rid, f in zip(manifest["id"], manifest["file"]):
    # ประมวลผลรายงาน 1 ฉบับ แล้วเก็บผลลงใน submission ภายใต้คีย์ rid (ทำงานจากในออกนอก)
    #   1) f"ocr/{f}"           : f-string ต่อชื่อโฟลเดอร์ ocr/ เข้ากับชื่อไฟล์ เป็นตำแหน่งไฟล์ในคลังข้อมูล
    #   2) lab_data(...)        : ดาวน์โหลดภาพรายงานนั้น คืนตำแหน่งไฟล์ในเครื่อง
    #   3) ocr_image(...)       : ทำ OCR ได้ข้อความ
    #   4) extract_report(...)  : สกัดผลการตรวจและรายการยา ได้ dict
    #   5) submission[rid] = ...: เก็บผลลงใน dict โดยใช้รหัสรายงานเป็นคีย์
    submission[rid] = extract_report(ocr_image(lab_data(f"ocr/{f}")))

# ----------------------------------------------------------------------------
# ส่วนที่ 6: แสดงตัวอย่างผล บันทึกเป็นไฟล์ JSON และส่งไปยังตารางอันดับ
# ----------------------------------------------------------------------------
# แสดงผลการสกัดของรายงาน R01 เป็นตัวอย่าง
#   - submission["R01"]  : ดึงผลของรายงานรหัส R01 จาก dict
#   - json.dumps(..., indent=1) : แปลงข้อมูลเป็นข้อความรูปแบบ JSON
#     - indent=1 : จัดย่อหน้า 1 ช่องในแต่ละระดับ ให้อ่านง่าย
print(json.dumps(submission["R01"], indent=1))
# เปิดไฟล์ lab07_submission.json เพื่อเขียนข้อมูล
#   - with ... as fh : เปิดไฟล์และตั้งชื่อเรียกว่า fh; เมื่อจบส่วนที่ย่อหน้าด้านล่าง ไฟล์จะถูกปิดให้อัตโนมัติ
#   - open(ชื่อไฟล์, "w") : เปิดไฟล์ในโหมด "w" (write = เขียนใหม่ทับของเดิม; ถ้ายังไม่มีไฟล์จะสร้างใหม่)
with open("lab07_submission.json", "w") as fh:
    # บันทึก submission ลงไฟล์ในรูปแบบ JSON
    #   - json.dump(ข้อมูล, ไฟล์, indent=1) : เขียนข้อมูลลงไฟล์ fh โดยจัดย่อหน้า 1 ช่อง
    #     (dump ไม่มี s = เขียนลงไฟล์, dumps มี s = แปลงเป็นข้อความ)
    json.dump(submission, fh, indent=1)
# แสดงข้อความยืนยัน พร้อมจำนวนรายงานในไฟล์
#   - f-string แทรกค่า len(submission) = จำนวนคีย์ (จำนวนรายงาน) ใน dict
print(f"สร้างไฟล์ lab07_submission.json เรียบร้อย ({len(submission)} รายงาน)")

# ตรวจว่ากรอก NAME และ JOIN_CODE ไว้ในเซลล์การตั้งค่าแล้วหรือไม่ (ข้อความว่างถือว่าเป็นเท็จ)
#   - and : ต้องกรอกครบทั้งสองค่า เงื่อนไขจึงเป็นจริง
if NAME and JOIN_CODE:
    # ถ้ากรอกครบ : ส่งไฟล์ไปยังตารางอันดับด้วยฟังก์ชันจากเซลล์การตั้งค่า และแสดงคะแนน/อันดับ
    submit_to_leaderboard("lab07_submission.json")
# else : มิฉะนั้น (ยังกรอกไม่ครบ)
else:
    # แสดงข้อความแนะนำวิธีส่งผลไปยังตารางอันดับ
    print("หากต้องการส่งผลไปยังตารางอันดับ โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่า แล้วรันเซลล์ดังกล่าวอีกครั้ง")

### แนวทางการพัฒนาให้ได้คะแนนสูงกว่า baseline

- **วิเคราะห์ข้อผิดพลาดก่อน:** แสดงข้อความ OCR ของรายงานบางฉบับ (`print(ocr_image(lab_data("ocr/reports/R05.png")))`) และเปรียบเทียบกับภาพต้นฉบับ
- **การเตรียมภาพก่อนทำ OCR (preprocessing):** แปลงเป็นภาพระดับเทา (grayscale) ขยายภาพ 2 เท่า (`img.resize(...)`) แปลงเป็นภาพสองระดับ (thresholding) และปรับภาพสแกนที่เอียงให้ตรง (deskew)
- **การปรับค่าพารามิเตอร์ของ Tesseract:** ทดลองโหมด `--psm` อื่น หรือ `-c tessedit_char_whitelist=...` สำหรับคอลัมน์ผลการตรวจ
- **การเพิ่มกฎแก้ไขข้อผิดพลาดจาก OCR** ใน `OCR_FIXES` (`S`↔`5`, `Z`↔`2`, `g`↔`9`) โดยตรวจสอบว่าไม่ทำให้ค่าที่ถูกต้องอยู่แล้วเกิดความผิดพลาด
- **การปรับปรุงรูปแบบการค้นหาผลการตรวจ:** ใช้รายการชื่อการตรวจที่กำหนดไว้ (ดังขั้นตอนที่ 3) หรือการจับคู่ชื่อที่สะกดผิดแบบใกล้เคียง (fuzzy matching, `difflib.get_close_matches`) ร่วมกับการตรวจสอบความสมเหตุสมผลของค่า
- **การสกัดรายการยา:** จับคู่กับพจนานุกรมยาโดยไม่คำนึงถึงตัวพิมพ์เล็กและใหญ่ หรือใช้โมเดล NER (ดูส่วนเสริม NER)


## ชิ้นงานที่ต้องส่ง (งานเดี่ยว)

ให้ส่ง **โน้ตบุ๊กนี้ซึ่งรันครบทุกเซลล์ตามลำดับจากบนลงล่าง** (Runtime ▸ Run all) ตามช่องทางที่รายวิชากำหนด โดยโน้ตบุ๊กต้องแสดง

- ตารางผลการตรวจที่สกัดได้พร้อมการระบุค่าผิดปกติ และรายการยาที่สกัดได้
- คำตอบ ✍️ ของคำถามที่ 1–4

ผู้เรียนที่เข้าร่วมโจทย์ท้าทายบนตารางอันดับ 🏆 สามารถระบุชื่อที่ใช้บนตารางอันดับและคะแนนสูงสุดไว้ในส่วนนี้ (ไม่บังคับ)

✍️ **ชื่อบนตารางอันดับ / คะแนน (ไม่บังคับ):**
